# RSNA Knee — 12 findings from one MRI study

A 2.5D DINOv2 baseline with report-derived weak labels, grouped folds, a runtime
guard, and resumable checkpoints.

**The shape of the problem.** Only 58 of 4,407 training studies carry official
labels. The other 4,349 carry a radiology report. `train.csv` has a `Report`
column and `test.csv` does **not** — text exists when fitting and is absent when
predicting. So reports can only ever be a source of *targets*, never a model
input. A text branch would have nothing to read at inference.

**What the metric changes.** Macro ROC-AUC is the unweighted mean of 12 per-label
AUCs, and AUC is invariant to any strictly increasing transform. Three
consequences drive design choices below: calibration is worthless (only rank
order matters), ensembles must average **ranks** not probabilities, and every
label costs the same — one label left at chance forfeits ~(M−0.5)/12 of the
score, so rare findings deserve *more* attention than common ones.

**Order of sections** follows what constrains what: config → targets → which
series to show the encoder → how to read pixels → model → training → OOF →
inference.

In [ ]:
# ── Section 0: environment ────────────────────────────────────────────────────
# Detects Kaggle vs local so the same file runs in both places. Locally it can
# only smoke-test shapes (there are 3 sample studies and no GPU); on Kaggle it
# trains for real.
import gc
import hashlib
import json
import math
import os
import random
import shutil
import tempfile
import time
import traceback
from dataclasses import dataclass, field, asdict, replace

import numpy as np
import pandas as pd

T_START = time.time()

ON_KAGGLE = os.path.exists("/kaggle/input")


def resolve_dir(candidates, must_contain=None):
    """First candidate that exists (and holds `must_contain`, if given).

    Kaggle mounts competitions at BOTH /kaggle/input/<comp> and
    /kaggle/input/competitions/<comp> depending on how the kernel was created, and
    Models at either /kaggle/input/<name>/... or /kaggle/input/models/<owner>/...
    Hard-coding one path is the single most common reason a CLI-pushed kernel dies
    instantly, so probe instead of assuming.
    """
    for c in candidates:
        if not c or not os.path.isdir(c):
            continue
        if must_contain and not os.path.exists(os.path.join(c, must_contain)):
            continue
        return c
    return None


if ON_KAGGLE:
    COMP = resolve_dir([
        "/kaggle/input/rsna-knee-abnormality-detection",
        "/kaggle/input/competitions/rsna-knee-abnormality-detection",
    ], must_contain="train.csv")
    WORK = "/kaggle/working"
    if COMP is None:
        print("!! competition data not found. /kaggle/input contains:")
        for root in ("/kaggle/input", "/kaggle/input/competitions"):
            if os.path.isdir(root):
                print(f"   {root}: {sorted(os.listdir(root))[:20]}")
        raise SystemExit("attach the competition to this kernel")
else:
    COMP = "data"
    WORK = "artifacts/local_run"


def print_input_layout(root="/kaggle/input", max_depth=3,
                       skip=("train_series", "test_series"), max_dirs=12):
    """Where did Kaggle mount things? A slug created today lays out /kaggle/input
    differently from one created last week (type-prefixed, one or two levels deeper), and
    a glob that is too shallow fails silently (traps 6f). Print the tree, minus the image
    trees, so the layout is read off the log instead of inferred after the fact."""
    if not os.path.isdir(root):
        return
    print(f"input layout under {root} (depth <= {max_depth}; image trees not descended):")

    def walk(d, depth):
        try:
            names = sorted(os.listdir(d))
        except OSError as e:
            print(f"  {d}: {e}")
            return
        dirs = [n for n in names if os.path.isdir(os.path.join(d, n))]
        files = [n for n in names if n not in dirs]
        print(f"  {d}: {len(dirs)} dirs, {len(files)} files"
              + (f"  e.g. {files[:4]}" if files else ""))
        if depth >= max_depth:
            return
        for n in dirs[:max_dirs]:
            if n in skip:
                print(f"  {os.path.join(d, n)}: (image tree, skipped)")
            else:
                walk(os.path.join(d, n), depth + 1)
        if len(dirs) > max_dirs:
            print(f"  {d}: ... {len(dirs) - max_dirs} more dirs not shown")

    walk(root, 0)


if ON_KAGGLE:
    print_input_layout()

os.makedirs(WORK, exist_ok=True)
print(f"ON_KAGGLE={ON_KAGGLE}  COMP={COMP}  WORK={WORK}")
if ON_KAGGLE:
    print(f"COMP contains: {sorted(os.listdir(COMP))[:12]}")

In [ ]:
# ── Section 1: configuration ──────────────────────────────────────────────────
# Everything tunable lives here so an experiment is one edit and the config is
# saved next to the checkpoints.
#
# `smoke` is the important one: it shrinks every dimension so the whole pipeline
# runs end to end in a couple of minutes. Never trust a long run you have not
# smoke-tested first — a crash in the inference cell after six hours of training
# costs a whole session.

LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
    "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture",
]

# Plane x acquisition slots, chosen so every finding has at least one sequence
# that shows it well: cruciates run obliquely (sagittal), collaterals and the
# meniscal body coronally, patellar cartilage axially.
SLOTS = [
    "SAG_FLUID_FS", "COR_FLUID_FS", "AX_FLUID_FS",
    "SAG_FLUID_NOFS", "COR_T1", "SAG_T1",
]


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ FORCE_SMOKE: True  = fast end-to-end check (minutes) -- use for the first │
# │                      run of any new/edited notebook.                     │
# │              False = real training run (hours, resumable).               │
# │              None  = auto (smoke locally, real on Kaggle).               │
# └──────────────────────────────────────────────────────────────────────────┘
FORCE_SMOKE = False

# ┌──────────────────────────────────────────────────────────────────────────┐
# │ MODE: "train" = train the configured folds, then infer if all complete.  │
# │       "infer" = load `{version}_fold*_best.pt` from a mounted kernel     │
# │                 output and only predict the test set. This is what gets  │
# │                 SUBMITTED: a code competition re-runs the notebook on    │
# │                 the hidden test, and re-training there would both blow   │
# │                 the runtime and change the model being scored.           │
# │       "oof_eval" = score each INFER_MEMBERS version's fold-0 checkpoint  │
# │                 on its held-out studies from the cache, with the TTA /  │
# │                 eval_windows in INFER_OVERRIDES -> {v}_fold0_tta_oof.csv │
# │                 for src/blend_check.py. No test prediction (P-12).       │
# │       "auto"  = "infer" if such checkpoints are mounted, else "train".   │
# └──────────────────────────────────────────────────────────────────────────┘
MODE = "auto"

# ┌──────────────────────────────────────────────────────────────────────────┐
# │ INFER_MEMBERS: versions rank-meaned in "infer" mode (P-21). Every        │
# │ mounted `{version}_fold*_best.pt` of every listed version is one member  │
# │ of a flat rank-mean. A listed version with NO mounted checkpoint is      │
# │ fatal, so the blend can never silently shrink to a model that was not   │
# │ the one validated (traps 6d). Empty -> [cfg.version]. Ignored in "train".│
# │ Members must share preprocessing geometry; head_type may differ.        │
# └──────────────────────────────────────────────────────────────────────────┘
# 2026-08-30: the seven-version default = submission #10, public LB 0.912 (fold-0 proxy OOF 0.8820).
# #9 without v09h = 0.909; #8 without the three c02 members = 0.900. Every version is a Dataset pin
# (kaggle/rsna-knee-infer/kernel-metadata.json); v09h picks up folds 1-4 automatically once shipped.
INFER_MEMBERS = ["v05a", "v05b", "v05g", "v06c", "v08w", "v10c", "v09h"]
# How members combine. "by_version": rank-mean the folds of each version, then rank-mean the
# versions -- every version gets one vote, however many folds it has. "flat": one vote per
# checkpoint. Measured on fold 0 (2026-08-29): attn + concat-8ep + concat-4ep flat = 0.8680,
# but with the concat-4ep version carrying 5 fold votes the flat mean drops to 0.8611 -- below
# the two-head blend alone (0.8670) -- because the attention head, the source of the
# diversity, becomes 1/7 of the vote. Versions are the unit of diversity; folds are replicates.
INFER_BLEND = "by_version"
# Per-version MEMBER-key overrides at inference (P-12 TTA for members whose checkpoints predate
# the fields, or an eval_windows cap). Only keys in INFER_MEMBER_KEYS are allowed -- an override
# can change how a member reads the decoded array, never which array is decoded. Example:
#   INFER_OVERRIDES = {"v05a": {"tta_offsets": (-1, 0, 1), "tta_pool": "focal"}}
INFER_OVERRIDES = {}

# ┌──────────────────────────────────────────────────────────────────────────┐
# │ ARMS: run several fold-0 configurations back to back in ONE session.     │
# │ Each arm gets its own version string, so its checkpoints and OOF csvs    │
# │ (`{version}_fold0_*`) never collide. An arm that raises is logged and    │
# │ skipped -- the session, not the code, is the scarce resource.            │
# │ Set ARMS = None for a single run of the plain config.                    │
# └──────────────────────────────────────────────────────────────────────────┘
# v11 measured the floor: |v04a - v04base| = 0.008 macro (up to 0.03 per label). Verdicts:
# jitter +0.011 KEEP; lat_undo -0.015 confirms P-05; attn -0.005 INCONCLUSIVE *because it had
# not converged* (still rising at ep3, train loss 0.447 vs 0.398). So the retest gives the head
# a schedule it can converge in, with a matched control that changes only the head.
# v13 (v05a attn / v05b concat, 8 ep) closed P-09 and gave the 0.896 two-head blend; the 5-fold
# v05g run showed folds add nothing on top of head diversity (#6/#7). P-10: the next member must
# make *different* errors -- a second architecture family. ConvNeXt-Tiny, concat head, jitter,
# 8 epochs under ckpt_policy=best_oof (unknown peak epoch for a CNN), backbone LR 1e-4 per the
# card (ImageNet-supervised CNN tolerates 5x the LR that DINOv2's SSL features need).
# 2026-08-30 (P-25 / P-26 / P-23 #2): members on the wide-band c02 cache with the window-attention
# head. `v08w` = DINOv2-S at 224 (isolates band + windows + head from resolution; ~2 h fold 0 on a
# T4). `v09h` = the timm CoAtNet-1 hybrid probe at 224 (RunPod). `v10c` = CoAtNet-2 @384, the 0.936
# notebook's strongest-member recipe (RunPod; grad_checkpoint for 24 GB cards, eval_windows 42 so
# the hidden-test rerun stays inside the budget -- oof_eval must use the same value).
C02 = {"cache_scheme": "c02", "window_mode": "random", "head_type": "window_attn",
       "train_windows": 24, "epochs": 8}
# 2026-09-21 (P-28): the PRODUCTION regime, copied from the public 0.924 member's training script:
# every report-labelled study is training data (no fold hold-out; the 58 gold rows are the only
# validation and are REPORTED, never selected on), 16 epochs, and `_best.pt` is the average of the
# EMA weights over the last three epochs (SWA) -- no epoch selection at all. Members trained this way
# have no OOF, so blend_check.py cannot judge them; their measure is gold-58 + the LB (P-27 fork).
# 2026-09-22 (P-29): 16 epochs over-train -- the fold-0 twin `v09p` peaked at epoch 8 (OOF 0.8731) and ended at 0.8607
# (11/12 labels down); SWA over the tail did not rescue it. Production members therefore train 8 epochs, SWA over 5-7.
PROD = {**C02, "epochs": 8, "train_all": True, "swa_last": 3, "ckpt_policy": "last"}
ARMS = [
    # 2026-09-23 (S2): the CoAtNet production member carries the S1 knobs -- two studies per BatchNorm batch (P-32,
    # `v09b` 0.8690) and light train-time augmentation (P-33, `v09c` 0.8730), both read against `v09h` 0.8683 on fold 0.
    # Both are under the 0.008 floor, both in the same direction, so both ride along by the pre-registered rule
    # (experiments.md 2026-09-23 "S1 A/B"). Training-only knobs: neither reaches inference (not INFER_MEMBER_KEYS).
    ("v09a", {**PROD, "backbone": "timm:coatnet_rmlp_1_rw_224", "img_size": 224, "lr_backbone": 1e-4,
              "batch_studies": 2, "grad_accum": 2, "aug": "light"}),
    ("v08a", {**PROD, "backbone": "dinov2", "img_size": 224}),
]
# Shipped fold-0 / 5-fold members (Datasets rsna-knee-ckpt-*) and finished probes: selectable through ARM_ONLY /
# RSNA_ARM for a rerun, but no longer run by default -- a forgotten sed would otherwise spend the
# session on arms that already exist before the production arm starts.
SHIPPED_ARMS = [
    ("v08w", {**C02, "backbone": "dinov2", "img_size": 224}),
    ("v09h", {**C02, "backbone": "timm:coatnet_rmlp_1_rw_224", "img_size": 224, "lr_backbone": 1e-4}),
    # P-29 epoch-budget probe (done 2026-09-22, train v21): the v09h recipe for 16 epochs, per-epoch OOF csvs.
    ("v09p", {**C02, "epochs": 16, "backbone": "timm:coatnet_rmlp_1_rw_224", "img_size": 224,
              "lr_backbone": 1e-4}),
    # S1 A/B (done 2026-09-23, train v23, one arm per GPU -- P-31 / P-32 / P-33). `v09b` = the v09h recipe with TWO
    # studies per BatchNorm batch (48 windows; grad_accum 2 keeps 4 studies per optimiser step, so windows/epoch and
    # the schedule are v09h's) -> fold-0 OOF 0.8690; `v09c` = v09b + light augmentation -> 0.8730; v09h 0.8683.
    ("v09b", {**C02, "backbone": "timm:coatnet_rmlp_1_rw_224", "img_size": 224, "lr_backbone": 1e-4,
              "batch_studies": 2, "grad_accum": 2}),
    ("v09c", {**C02, "backbone": "timm:coatnet_rmlp_1_rw_224", "img_size": 224, "lr_backbone": 1e-4,
              "batch_studies": 2, "grad_accum": 2, "aug": "light"}),
]
ARM_V10C = ("v10c", {**C02, "backbone": "timm:coatnet_rmlp_2_rw_384", "img_size": 384,
                     "lr_backbone": 1e-4, "eval_windows": 42, "grad_checkpoint": True})
PRIMARY_ARM = "v09a"
ARM_FOLDS = (0,)
# Sed'd per kernel at build time (like FIVE_FOLD / STACK_RUN below, and mutually exclusive with
# them): run exactly ONE arm and make it PRIMARY_ARM, so rsna-knee-train and rsna-knee-folds can
# each take one production arm in the same sitting (two 16-epoch arms never fit one 9 h session):
#   sed 's/^ARM_ONLY = ""/ARM_ONLY = "v08a"/' src/kaggle_pipeline.py > artifacts/train_v08a.py
ARM_ONLY = ""
# Off-Kaggle runner (scripts/runpod_bootstrap.sh): RSNA_ARM=<version> does the same through the
# environment; RSNA_WORKERS / RSNA_RUNTIME_H override the loader worker count and the session
# guard. One filter serves both; the environment wins when both are set.
_only = os.environ.get("RSNA_ARM") or ARM_ONLY
if _only:
    ARMS = [a for a in list(ARMS) + list(SHIPPED_ARMS) + [ARM_V10C] if a[0] == _only]
    if not ARMS:
        raise SystemExit(f"arm {_only!r} is not one of the defined arms")
    PRIMARY_ARM = _only
    print(f"{'RSNA_ARM' if os.environ.get('RSNA_ARM') else 'ARM_ONLY'}: running only {_only}")

# Refuse to silently train the v02 decode path when the cache is expected (traps 6f).
ALLOW_DECODE_FALLBACK = False

# Flipped by sed for kaggle/rsna-knee-folds: five folds of the confirmed v04d recipe
# (concat + jitter, 4 epochs) for the first real ensemble. 5 x 4 epochs ~= 4.5 h; 5 x 8 would
# be ~9 h and needs the resume path instead.
# `v05f` is RETIRED: rsna-knee-folds v2 wrote v05f_fold*.pt trained on the v02 decode path
# (the cache never mounted, traps 6f). Never mount that output; the valid re-run is `v05g`.
FIVE_FOLD = False
if FIVE_FOLD:
    ARMS = [("v05g", {"cache_jitter": True, "folds": (0, 1, 2, 3, 4), "epochs": 4})]
    PRIMARY_ARM = "v05g"

# Flipped by sed for kaggle/rsna-knee-stack (P-23 candidate #3): five folds of the 16-channel
# member, 8 epochs under best_oof. It has its OWN kernel slug so pushing it never repoints the
# rsna-knee-train / rsna-knee-folds mounts that rsna-knee-infer reads (handoff 2026-08-30).
STACK_RUN = False
if STACK_RUN:
    ARMS = [("v07s", {"stack_mode": "channels", "cache_jitter": True,
                      "folds": (0, 1, 2, 3, 4), "epochs": 8})]
    PRIMARY_ARM = "v07s"

# ┌──────────────────────────────────────────────────────────────────────────┐
# │ PARALLEL_ARMS (P-31, 2026-09-22): Kaggle's "NvidiaTeslaT4" machine is    │
# │ GPU T4 x2 (a single T4 is not offered; kaggle-cli docs PR #1198) and the │
# │ weekly quota charges session hours -- every training session so far     │
# │ trained on cuda:0 with the second T4 idle. Sed'd at build like ARM_ONLY: │
# │   sed 's/^PARALLEL_ARMS = ()/PARALLEL_ARMS = ("v09b", "v09c")/' ...      │
# │ Section 8 then runs one CHILD PROCESS per arm, one GPU each, this very   │
# │ file as the child's script (RSNA_CHILD=1, RSNA_ARM=<arm>,                │
# │ CUDA_VISIBLE_DEVICES=<i>, RSNA_TRAIN_ONLY=1), each writing <arm>.log.    │
# │ nbgen embeds the pipeline text below (zlib + base64 + sha256) so the     │
# │ notebook can hand itself to the children; a .py run uses __file__.       │
# │ Exclusive with ARM_ONLY / FIVE_FOLD / STACK_RUN. () = sequential loop.   │
# └──────────────────────────────────────────────────────────────────────────┘
PARALLEL_ARMS = ("v09a", "v08a")
SELF_SOURCE_SHA256 = '997ef75a4b0a1712eae0334fdfc89362455aab5b322691441e9878e39eb0483c'
SELF_SOURCE_B64 = (
    '...'
)
if PARALLEL_ARMS and not os.environ.get("RSNA_CHILD"):
    if ARM_ONLY or FIVE_FOLD or STACK_RUN:
        raise SystemExit("PARALLEL_ARMS is exclusive with ARM_ONLY / FIVE_FOLD / STACK_RUN")
    _known = {a[0] for a in list(ARMS) + list(SHIPPED_ARMS) + [ARM_V10C]}
    _bad = [a for a in PARALLEL_ARMS if a not in _known]
    if _bad:
        raise SystemExit(f"PARALLEL_ARMS {_bad} not among the defined arms {sorted(_known)}")
    print(f"PARALLEL_ARMS: {list(PARALLEL_ARMS)} (one child process per GPU; this process only launches and waits)")


@dataclass
class Config:
    smoke: bool = field(default_factory=lambda:
                        (not ON_KAGGLE) if FORCE_SMOKE is None else bool(FORCE_SMOKE))
    version: str = "v03"             # v01 rank targets (smoke only) · v02 prob targets, decode per epoch · v03 from cache

    # data
    img_size: int = 224              # DINOv2 ViT-S/14 patches 14 -> 224 = 16x16 tokens
    slices_per_slot: int = 6         # uniformly sampled centres per slot
    triplet_gap: int = 2             # channels are slices [i-gap, i, i+gap]  (decode path only)

    # Cache path (P-01). When a cache built by src/cache_pipeline.py is mounted, training
    # reads one uint8 array per study; TEST studies are built on the fly by the very same
    # functions (crop, per-series normalisation, laterality), so train and test share one
    # preprocessing code path. Triplets are neighbouring cached slices [c-1, c, c+1].
    use_cache: bool = True
    cache_n_slices: int = 16         # stored slices per slot (must match the mounted cache)
    cache_px: int = 224
    crop_mm: float = 130.0
    lat_dead_zone_mm: float = 20.0
    # P-05 ablation. The cache stores every knee in a canonical left-knee frame; this puts
    # the right knees back into their own chirality at load time (both cache operations are
    # involutions), so laterality can be ablated without rebuilding 21 GB of cache.
    lat_undo: bool = False
    # P-08 sub-arm: jitter the K sampled slice centres by +-1 cached slice each epoch. The
    # only real augmentation this pipeline has (the other is Gaussian noise at sigma 0.01).
    cache_jitter: bool = False
    # P-23 candidate #3: how the 16 cached slices of a slot reach the encoder. "triplet" = K
    # centres, each a 3-channel [c-1, c, c+1] image (v03..v06). "channels" = ONE image per slot
    # with all 16 cached slices as its input channels -- the whole stack in one forward pass, a
    # different input representation from every triplet member (the 0.936 notebook's second
    # family works this way). The patch-embedding conv is widened 3 -> 16 (RGB-mean weights
    # x 3/16, response scale preserved) and trained at `lr_stem`. 6 encoder passes per study
    # instead of 36, so an epoch is ~6x cheaper. With `cache_jitter` the whole stack shifts +-1.
    stack_mode: str = "triplet"
    lr_stem: float = 2e-4            # channels mode only: the widened patch-embedding conv

    # Cache SCHEME (2026-08-30). "c01" = the original cache: dense [6, 16, 224, 224] per study,
    # one .npy each, per-plane band sag 8-92 / cor 20-80 / ax 10-90 -- described by cache_px /
    # cache_n_slices above. "c02" = the wide-band rebuild: the same six slots with RAGGED slice
    # budgets (18/12/12/14/8/8 = 72 slices, order = SLOTS), band 2-98 % for every plane, 336 px,
    # stored FLAT [72, 336, 336] inside multi-study blob files. Why: the 0.936 notebook's best
    # member uses 2-98 % and reports the outer slices carry the collaterals and the lateral
    # meniscus -- our two weakest labels. Both caches can be mounted at once; each Config resolves
    # to exactly one of them through cache_version_for(). The c02 fields below are ignored for c01.
    cache_scheme: str = "c01"
    cache_px_wide: int = 336         # c02 stored resolution (cache_px stays the c01 value)
    cache_slot_slices: tuple = ()    # c02 budgets per slot; () -> (18, 12, 12, 14, 8, 8)
    cache_band: tuple = ()           # c02 (lo, hi) for every plane; () -> (0.02, 0.98)

    # WINDOWS (P-25). "fixed" = K equidistant triplet centres per slot (every member through
    # v06c; array_to_tensor). "random" = the study is a set of (slot, centre) windows: training
    # samples `train_windows` of them (stratified, >= 2 per present slot) as its augmentation,
    # evaluation feeds every valid window (or `eval_windows` equidistant ones when > 0 -- the
    # SAME value must be used by oof_eval and infer so the OOF number predicts the LB number).
    # The Dataset ships the uint8 array + indices; the model gathers/normalises/resizes on the
    # GPU, so 60 windows never travel through DataLoader shared memory as float tensors.
    window_mode: str = "fixed"
    train_windows: int = 24
    eval_windows: int = 0
    # P-33 (2026-09-22). Train-time augmentation of the gathered windows, on the GPU, window mode only.
    # "none" = today's path bit for bit (the Gaussian noise at sigma 0.01, p 0.5 stays and draws the same
    # RNG). "light" = per window at p 0.8: affine (rotation +-8 deg, zoom-in 1.00-1.08, shift +-5 %, zero
    # padding), then gamma 0.8-1.25 and gain 0.9-1.1, clamped to [0, 1], all before the ImageNet
    # normalisation. No flips: medial != lateral (P-05). Training-only -- deliberately NOT an
    # INFER_MEMBER_KEY, so a checkpoint's saved `aug` never reaches inference.
    aug: str = "none"
    # Slice-offset TTA for fixed-window members (P-12): the K centres are shifted by each offset
    # (clipped to the stack), one forward per offset, probabilities pooled per label.
    # tta_pool "mean" = average; "focal" = the 0.936 notebook's rule: max over views for
    # Fracture / Contusion / both Menisci / Baker's, top-2 mean for ACL / MCL, mean otherwise.
    tta_offsets: tuple = (0,)
    tta_pool: str = "mean"

    # model
    # P-10: a second architecture family as a blend member. "dinov2" = DINOv2 ViT-S/14 (CLS
    # token); "convnext_tiny" = HF facebook/convnext-tiny-224 (ImageNet-1k, Apache-2.0,
    # LayerNorm throughout so batch-of-1 is safe; pooled 768-d output). Same 224x3 ImageNet-
    # normalised triplets feed both, so a study array is shared across families at inference.
    backbone: str = "dinov2"
    backbone_dir: str = ""           # resolved from `backbone` below (and per arm / per member)
    dropout: float = 0.1
    # P-09. "concat" = v03 baseline (6 slot vectors + mask -> one Linear); "attn" = 12
    # learned label queries doing masked attention over the present slot vectors.
    # P-25. "window_attn" = 12 label queries attending over EVERY (slot, window) token of the
    # study (per-label softmax over windows, slot embedding added), with no label-agnostic
    # per-slot pooling in between -- the 0.936 notebook's strongest member pools this way.
    head_type: str = "concat"
    slot_dropout: float = 0.0        # P-09 sub-arm; 0 keeps the head A/B clean
    slot_embed: bool = True          # window_attn: add a learned per-slot embedding to each token
    # timm hybrids (P-23 #2): `backbone="timm:<arch>"` loads <dir>/model.safetensors offline.
    # Gradient checkpointing halves activation memory for coatnet_2 @384 x 24 windows on 24 GB.
    grad_checkpoint: bool = False

    # optimisation
    folds: tuple = (0, 1, 2, 3, 4)
    epochs: int = 8         # v11: with jitter the OOF curve had not peaked by epoch 3
    lr_head: float = 1e-3
    # Backbone LR and layer-wise decay (P-03). Every medical DINOv2 fine-tuning
    # recipe we found lands at 1e-6..2e-5 for the top block; a uniform 5e-5 is the
    # regime described as catastrophic forgetting of the self-supervised features.
    # Block i gets lr_backbone * llrd_decay ** (n_blocks - 1 - i); the patch/pos
    # embeddings get one more decay step. 0.75 is the BEiT/MAE convention.
    lr_backbone: float = 2e-5
    llrd_decay: float = 0.75
    weight_decay: float = 0.02       # not applied to biases / LayerNorm
    # EMA of the weights is what gets validated and saved (robust to label noise,
    # and makes fixed-epoch selection safe). 0 disables.
    ema_decay: float = 0.998
    # Studies per DataLoader batch. Fixed-window members: one study = up to 6 slots x 6 slices of ViT work.
    # Window mode (P-32, 2026-09-22): > 1 concatenates the studies' sampled windows into ONE encoder pass
    # (collate_windows), so a BatchNorm backbone (timm CoAtNet's MBConv stages) normalises over several
    # studies instead of 24 windows of one; the loss stays per-study normalised. Evaluation and inference
    # always run one study per batch (not an INFER_MEMBER_KEY). Pair with grad_accum so studies per
    # optimiser step stay comparable across arms (v09h: 1 x 4; v09b: 2 x 2).
    batch_studies: int = 1
    grad_accum: int = 4
    warmup_frac: float = 0.1
    max_grad_norm: float = 1.0
    amp: bool = True

    # supervision
    gold_weight: float = 8.0
    weak_weight_floor: float = 0.15

    # runtime
    runtime_limit_hours: float = float(os.environ.get("RSNA_RUNTIME_H", 8.3))   # headroom under Kaggle's 9 h
    seed: int = 42
    num_workers: int = int(os.environ.get("RSNA_WORKERS", 2))     # 8 on a local-NVMe box
    # Which epoch `_best.pt` holds. "best_oof": the epoch with the highest OOF-vs-teacher
    # macro-AUC so far (P-22: +0.013 split-half for the concat head, ~0 for attn, gold flat).
    # "last": EMA weights after the last completed epoch (fixed-epoch, used through v05).
    ckpt_policy: str = "best_oof"
    # Production regime (P-28, 2026-09-21; the public 0.924 member's recipe): train on EVERY
    # report-labelled study and hold out nothing but the 58 gold rows, which are reported per epoch
    # and never selected on. One "fold" named fold0, so `{version}_fold0_best.pt` is what
    # rsna-knee-infer globs. Requires ckpt_policy="last": "best_oof" would pick the epoch on
    # gold-58 (Hanley-McNeil SE ~0.04 macro), which stays banned.
    train_all: bool = False
    # > 0: keep the EMA state_dict of the last N COMPLETED epochs in host RAM (persisted in _last.pt,
    # so a resumed session averages the same N) and write their element-wise mean as _best.pt;
    # the final-epoch EMA is kept as `_lastema.pt` for the A/B. 0 = plain ckpt_policy.
    swa_last: int = 0
    # Smoke only: cap the header scan so a verification run does not spend minutes
    # reading all ~24k series headers before it reaches the training loop.
    smoke_max_studies: int = 24

    def __post_init__(self):
        if self.cache_scheme not in ("c01", "c02"):
            raise SystemExit(f"unknown cache_scheme {self.cache_scheme!r}")
        if self.cache_scheme == "c02":
            self.cache_slot_slices = tuple(self.cache_slot_slices) or (18, 12, 12, 14, 8, 8)
            self.cache_band = tuple(self.cache_band) or (0.02, 0.98)
            if self.stack_mode != "triplet" or self.lat_undo:
                raise SystemExit("stack_mode='channels' and lat_undo are c01-only (v07s is dead, "
                                 "P-05 is closed); they were not ported to the flat c02 layout")
        self.tta_offsets = tuple(self.tta_offsets)
        if self.aug not in ("none", "light"):
            raise SystemExit(f"unknown aug {self.aug!r} (none | light)")
        if self.aug != "none" and self.window_mode != "random":
            raise SystemExit("aug runs inside forward_windows only: set window_mode='random' (a fixed-window arm "
                             "would otherwise claim an augmentation that never runs)")
        if self.batch_studies > 1 and self.window_mode == "random" and self.cache_scheme != "c02":
            raise SystemExit("batch_studies > 1 in window mode needs the flat c02 cache (no c01 window member exists)")
        if self.smoke:
            self.folds = (0,)
            self.epochs = 1
            self.slices_per_slot = 2
            if not os.environ.get("RSNA_SMOKE_FULL_WINDOWS"):
                # RSNA_SMOKE_FULL_WINDOWS=1 keeps the real window count so a Kaggle smoke exercises the
                # batch_studies x train_windows memory path (P-32) on a handful of studies
                self.train_windows = 4
            if not str(self.backbone).startswith("timm:"):
                # a fixed-resolution timm hybrid (coatnet_rmlp_2_rw_384) crashes at 224; DINOv2
                # and ConvNeXt take any size, and 224 keeps a CPU smoke fast
                self.img_size = 224
            self.runtime_limit_hours = 0.4
            self.ema_decay = 0.9      # 8 steps of smoke would leave a 0.998 EMA ~= init
        # After the smoke block on purpose: smoke's epochs=1 clamps swa_last to 1, so the SWA
        # save / load / evaluate path is still exercised (a mean of one snapshot is the identity).
        if self.train_all:
            self.folds = (0,)            # one pass, named fold0 (checkpoint glob + ARM_FOLDS agree)
            if self.ckpt_policy != "last":
                raise SystemExit("train_all=True needs ckpt_policy='last' (best_oof would pick the "
                                 "epoch on the 58 gold rows)")
        if self.swa_last > 0:
            self.swa_last = min(int(self.swa_last), int(self.epochs))
            if self.ema_decay <= 0:
                raise SystemExit("swa_last averages EMA snapshots; set ema_decay > 0")


CACHE_BAND ={"Sagittal": (0.08, 0.92), "Axial": (0.10, 0.90), "Coronal": (0.20, 0.80)}
PLANE_OF_SLOT = {"SAG_FLUID_FS": "Sagittal", "COR_FLUID_FS": "Coronal", "AX_FLUID_FS": "Axial",
                 "SAG_FLUID_NOFS": "Sagittal", "COR_T1": "Coronal", "SAG_T1": "Sagittal"}
CACHE_PCT = (1.0, 99.0)      # per-series percentile window (the cache builder's pct_lo / pct_hi)


def cache_version_of(scheme, px, slot_slices, band, crop_mm, lat_dead_zone_mm):
    """Name of the directory a cache lives in. It must encode EVERYTHING that changes the
    stored bytes: c01's string left out the band and the percentiles, so a band change at the
    same px/slices would have been silently accepted by the loader (traps 23). Byte-identical
    copy in src/kaggle_pipeline.py -- src/cache_selftest.py asserts the two agree."""
    if scheme == "c01":
        return f"c01_p{px}_s{slot_slices[0]}_crop{int(crop_mm)}_lat{int(lat_dead_zone_mm)}"
    lo, hi = band["Sagittal"]                       # c02: one band for every plane
    return (f"c02_p{px}_b{'-'.join(str(int(s)) for s in slot_slices)}"
            f"_band{int(round(lo * 100))}-{int(round(hi * 100))}"
            f"_crop{int(crop_mm)}_lat{int(lat_dead_zone_mm)}")


def slot_offsets(slot_slices):
    """Start index of each slot inside the flat (sum(slot_slices), P, P) array, plus the total."""
    starts, acc = [], 0
    for n in slot_slices:
        starts.append(acc)
        acc += int(n)
    return tuple(starts), acc


def _cfg_get(c):
    """Uniform reader over a Config object or a checkpoint's saved-config dict (old checkpoints
    lack the new fields, so every read carries the c01-era default)."""
    if isinstance(c, dict):
        return lambda k, d=None: c.get(k, d)
    return lambda k, d=None: getattr(c, k, d)


def cache_geom(c):
    """(scheme, px, slot_slices, band_dict) that Config `c` resolves to -- the one place the two
    schemes' field conventions meet. Works on a Config or on a saved-config dict."""
    g = _cfg_get(c)
    scheme = g("cache_scheme", "c01")
    if scheme == "c01":
        n = int(g("cache_n_slices", 16))
        return "c01", int(g("cache_px", 224)), (n,) * len(SLOTS), dict(CACHE_BAND)
    ss = tuple(int(s) for s in (g("cache_slot_slices", ()) or (18, 12, 12, 14, 8, 8)))
    band = tuple(float(b) for b in (g("cache_band", ()) or (0.02, 0.98)))
    return "c02", int(g("cache_px_wide", 336)), ss, {p: band for p in ("Sagittal", "Coronal", "Axial")}


def cache_version_for(c):
    g = _cfg_get(c)
    scheme, px, ss, band = cache_geom(c)
    return cache_version_of(scheme, px, ss, band, float(g("crop_mm", 130.0)),
                            float(g("lat_dead_zone_mm", 20.0)))


cfg = Config()
CACHE_VERSION = cache_version_for(cfg)     # the DEFAULT config's cache; arms/members recompute
# cache_version -> {StudyInstanceUID -> locator}; a locator is a .npy path (c01, one study per
# file) or (blob_path, row) (c02). Filled per cache version in Section 8 / at inference.
CACHE_INDEX = {}

# Weight locations differ between Kaggle (mounted Model, two possible layouts) and
# local (models/). config.json is the marker that a real HF checkpoint dir is there.
BACKBONES = {
    "dinov2": ([
        "/kaggle/input/dinov2/pytorch/small/1",
        "/kaggle/input/models/metaresearch/dinov2/pytorch/small/1",
        "/kaggle/input/dinov2-small/pytorch/small/1",
        "models/dinov2_small",
    ], "metaresearch/dinov2 PyTorch/small/1 as a Model input"),
    "convnext_tiny": ([
        "/kaggle/input/datasets/tiankljucanin/convnext-tiny-224-hf",
        "/kaggle/input/convnext-tiny-224-hf",
        "models/convnext_tiny",
    ], "tiankljucanin/convnext-tiny-224-hf as a Dataset input"),
    # timm hybrids (P-23 #2). Each Dataset holds the HF timm repo files: config.json (the marker
    # resolve_dir probes) + model.safetensors; timm itself ships in the Kaggle image.
    "timm:coatnet_rmlp_1_rw_224": ([
        "/kaggle/input/datasets/tiankljucanin/timm-coatnet-rmlp-1-rw-224",
        "/kaggle/input/timm-coatnet-rmlp-1-rw-224",
        "models/coatnet_rmlp_1_rw_224",
    ], "tiankljucanin/timm-coatnet-rmlp-1-rw-224 as a Dataset input"),
    "timm:coatnet_rmlp_2_rw_384": ([
        "/kaggle/input/datasets/tiankljucanin/timm-coatnet-rmlp-2-rw-384",
        "/kaggle/input/timm-coatnet-rmlp-2-rw-384",
        "models/coatnet_rmlp_2_rw_384",
    ], "tiankljucanin/timm-coatnet-rmlp-2-rw-384 as a Dataset input"),
}


def resolve_backbone_dir(backbone: str) -> str:
    """HF checkpoint dir for a backbone family; both mount layouts probed (traps 6f/10)."""
    if backbone not in BACKBONES:
        raise SystemExit(f"unknown backbone {backbone!r}; known: {sorted(BACKBONES)}")
    candidates, attach = BACKBONES[backbone]
    d = resolve_dir(candidates, must_contain="config.json")
    if d is None:
        raise SystemExit(f"{backbone} weights not found -- attach {attach}")
    return d


cfg.backbone_dir = resolve_backbone_dir(cfg.backbone)
print(f"backbone: {cfg.backbone} @ {cfg.backbone_dir}")
print(json.dumps({k: str(v) for k, v in asdict(cfg).items()}, indent=1))


def seed_all(s: int) -> None:
    random.seed(s)
    np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s)
        torch.cuda.manual_seed_all(s)
    except Exception:
        pass


seed_all(cfg.seed)


def elapsed_h() -> float:
    return (time.time() - T_START) / 3600.0


def out_of_time() -> bool:
    """Runtime guard. Five folds do not fit in one 9 h session, so training must be
    able to stop cleanly and resume in the next session rather than be killed."""
    return elapsed_h() > cfg.runtime_limit_hours

## Section 2: where the targets come from

The reports are the only way to supervise 4,349 studies, and reading them well
is a multilingual NLP problem (~9–12 languages, and for several findings *most*
mentions are negative because a report lists what was checked and found intact).

Rather than rebuild a lexicon, this mounts the public LLM-read label tables and
averages their probabilities. Measured gold macro-AUC (n=58): hans_v4 0.893,
pilkwang 0.870, sol56 0.835, blend 0.895 (rank blend 0.893 -- same within noise,
but the rank blend put confident negatives at ~0.3 instead of ~0; see P-00 in
docs/proposals.md).

Two details matter more than the blend:

1. **Grade the mention, don't binarise it.** The reporting radiologist and the
   annotator do not share a threshold — a report saying *small joint effusion*
   can sit against a negative annotation, because annotators marked only
   findings they judged significant and graded "on the fence" as negative. So
   `term present ⇒ positive` is wrong by construction. Soft targets cost nothing
   because only rank order is read.
2. **Weight by how confidently the report could be read.** Source disagreement and
   indecisiveness both lower the weight. Measured caveat: a report that never mentions
   synovitis blends to ~0.18 and is *not* strongly down-weighted (0.69 vs 0.80 on
   addressed rows) — silence looks like a confident negative. Open card P-07/P-16.

The 58 official labels overwrite the weak ones and carry `gold_weight`.

In [ ]:
# ── Section 2: targets ────────────────────────────────────────────────────────
LLM_SOURCES = [
    ("hans_v4", [
        "/kaggle/input/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv",
        "data/llm_labels/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv",
    ]),
    ("pilkwang", [
        "/kaggle/input/rsna-knee-llm-labels/report_labels_v2.csv",
        "data/llm_labels/rsna-knee-llm-labels/report_labels_v2.csv",
    ]),
    ("sol56", [
        "/kaggle/input/rsna-knee-llm-report-labels-sol56/labels_llm_gpt56sol.csv",
        "data/llm_labels/rsna-knee-llm-report-labels-sol56/labels_llm_gpt56sol.csv",
    ]),
]


def shallow_glob(root, name, max_depth=3, skip=("train_series", "test_series")):
    """`glob` for `name` at depth 1..max_depth below `root` WITHOUT descending into the
    image trees. A recursive `**` glob over /kaggle/input walks ~819k DICOM files on a
    network mount -- minutes of dead time on every run, invisible on the rerun."""
    import glob
    hits = []
    for d in range(0, max_depth + 1):          # depth 0 = directly under root
        pat = os.path.join(root, *(["*"] * d), name)
        hits += [h for h in glob.glob(pat)
                 if not any(f"{os.sep}{sk}{os.sep}" in h or f"/{sk}/" in h for sk in skip)]
    return sorted(hits)


def first_existing(paths):
    """Exact candidates first, then search /kaggle/input for the filename.

    Dataset mount slugs are predictable but not guaranteed, so fall back to finding
    the file by name rather than failing and silently training on prior-only targets.
    """
    for p in paths:
        if os.path.exists(p):
            return p
    if ON_KAGGLE:
        want = os.path.basename(paths[0])
        for hit in shallow_glob("/kaggle/input", want, max_depth=4):
            return hit
    return None


def auc_score(y, s) -> float:
    """Mann-Whitney AUC, hand-rolled so the notebook needs no sklearn."""
    y = np.asarray(y)
    s = np.asarray(s, dtype=float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    npos, nneg = int((y == 1).sum()), int((y == 0).sum())
    if npos == 0 or nneg == 0:
        return float("nan")
    r = pd.Series(s).rank().to_numpy()
    return float((r[y == 1].sum() - npos * (npos + 1) / 2) / (npos * nneg))


def build_targets(train_csv: str):
    tr = pd.read_csv(train_csv)
    idx = pd.Index(tr.StudyInstanceUID)
    is_gold = tr[LABELS].notna().all(axis=1)

    loaded = {}
    for name, paths in LLM_SOURCES:
        p = first_existing(paths)
        if p is None:
            print(f"  ! {name}: not mounted, skipping")
            continue
        d = pd.read_csv(p).set_index("StudyInstanceUID").reindex(idx)
        if set(LABELS) <= set(d.columns):
            loaded[name] = d
            print(f"  loaded {name} from {p}")

    soft = pd.DataFrame(index=idx)
    wt = pd.DataFrame(index=idx)
    if loaded:
        for lab in LABELS:
            arr = np.vstack([d[lab].to_numpy(dtype=float) for d in loaded.values()])
            # Probability space, NOT rank space (P-00). Rank-percentiles give tied
            # values their average rank, so on a label where most reports say exactly
            # 0 every confident negative landed at ~0.3-0.4 while gold rows sit at a
            # hard 0/1. BCE fits the value, not the order. Ranks are for scoring and
            # for ensembling predictions, never for building a target.
            with np.errstate(invalid="ignore"):
                soft[lab] = np.nanmean(arr, axis=0)
                spread = np.nanstd(arr, axis=0)
                mean = np.nanmean(arr, axis=0)
            agree = 1.0 - np.nan_to_num(spread, nan=0.5) * 2.0
            decisive = np.abs(np.nan_to_num(mean, nan=0.5) - 0.5) * 2
            wt[lab] = np.clip(0.5 * np.clip(agree, 0, 1) + 0.5 * np.clip(decisive, 0, 1),
                              cfg.weak_weight_floor, 1.0)
    else:
        # No label tables mounted: fall back to prior-only targets so the pipeline
        # still runs. This trains nothing useful and says so loudly.
        print("  ! NO LLM LABELS MOUNTED — using prior-only targets (smoke only)")
        for lab in LABELS:
            soft[lab] = 0.5
            wt[lab] = cfg.weak_weight_floor

    gold = tr.set_index("StudyInstanceUID")[LABELS]

    # Score the teacher BEFORE the gold override, otherwise we are grading the gold
    # labels against themselves and always get 1.000.
    gold_pos = is_gold.to_numpy()
    teacher_auc = float("nan")
    if loaded and gold_pos.sum():
        gy = gold.loc[idx[gold_pos]].astype(float)
        a = [auc_score(gy[l].to_numpy(), soft.loc[gold_pos, l].to_numpy())
             for l in LABELS]
        teacher_auc = float(np.nanmean(a))
        print(f"  teacher (report labels only) gold macro-AUC: {teacher_auc:.4f}")
        print("  ^ this is the signal ceiling the vision model is distilling from")

    for lab in LABELS:
        g = gold[lab].reindex(idx)
        have = g.notna().to_numpy()
        soft.loc[have, lab] = g[have].to_numpy()
        wt.loc[have, lab] = cfg.gold_weight

    for lab in LABELS:
        m = soft[lab].isna()
        if m.any():
            soft.loc[m, lab] = float(soft[lab].mean())
            wt.loc[m, lab] = cfg.weak_weight_floor

    # ---- folds: group studies that share a report text -------------------
    # 49 report texts are shared by 183 studies (largest group 37). Studies sharing
    # a report share a target vector, so splitting them across folds leaks the
    # answer into validation.
    norm = tr.Report.fillna("").str.strip().str.lower()
    grp = norm.map(lambda t: hashlib.md5(t.encode("utf-8")).hexdigest()[:16])
    meta = pd.DataFrame({
        "StudyInstanceUID": tr.StudyInstanceUID.to_numpy(),
        "is_gold": is_gold.astype(int).to_numpy(),
        "report_group": grp.to_numpy(),
    })
    g = meta.groupby("report_group").agg(n=("StudyInstanceUID", "size"),
                                         gold=("is_gold", "sum"))
    g = g.sample(frac=1.0, random_state=cfg.seed).sort_values(
        ["gold", "n"], ascending=False)
    n_folds = 5
    sizes = np.zeros(n_folds)
    golds = np.zeros(n_folds)
    assign = {}
    for gid, row in g.iterrows():
        # Balance gold first (so every fold is scoreable), then total size.
        k = int(np.lexsort((sizes, golds))[0]) if row.gold > 0 else int(np.argmin(sizes))
        assign[gid] = k
        sizes[k] += row.n
        golds[k] += row.gold
    meta["fold"] = meta.report_group.map(assign)

    tgt = soft.reset_index(drop=True)
    tgt.columns = LABELS
    wdf = wt.reset_index(drop=True)
    wdf.columns = [f"w__{c}" for c in LABELS]
    out = pd.concat([meta.reset_index(drop=True), tgt, wdf], axis=1)

    print(f"  targets: {out.shape[0]} studies, {int((out.is_gold == 1).sum())} gold")
    print("  fold sizes:",
          out.groupby("fold").size().to_dict(),
          "gold:", out.groupby("fold").is_gold.sum().to_dict())
    return out


targets = build_targets(os.path.join(COMP, "train.csv"))
if not os.environ.get("RSNA_CHILD"):      # P-31 children would be two concurrent writers of the same bytes
    targets.to_csv(os.path.join(WORK, "targets.csv"), index=False)
targets.head(3)

## Section 3: which series to show the encoder

A study holds 3–14 series (median 5) in three planes. The encoder cannot see all
of them, so each study is reduced to at most six slots.

`train_series.csv` ships `Fluid_Sensitive` and `Fat_Suppression`, but **as
delivered they carry one bit, not two** — verified on the full training set: only
`(1,1)` (14,010 rows) and `(0,0)` (10,361) ever occur, never a mixed pair. Two
physically independent properties collapsed into one axis. Fluid sensitivity is a
property of the *contrast weighting* (set by TR/TE); fat suppression is a
*preparation* applied on top of any weighting. So both are recovered from the
DICOM headers.

`Anatomical_Plane`, by contrast, **is** trustworthy — it agreed 100% with the
plane derived from `ImageOrientationPatient` on the sample studies, so it is used
as-is and only recomputed when missing.

Slot matching runs in two tiers. Strict (right plane, fluid **and** fat-sat) left
2 of 12 sample series unassigned and one study at 2/6 slots, because real studies
routinely carry an axial fluid series with no fat suppression. A relaxed second
tier lifted that to 4/6 and 5/6.

In [ ]:
# ── Section 3: series selection ───────────────────────────────────────────────
import pydicom

TR_SHORT_MAX = 800.0   # ms
TE_LONG_MIN = 60.0     # ms
FATSAT_TOKENS = ("fs", "fatsat", "fat_sat", "stir", "spir", "spair", "tirm",
                 "dixon", "chess", "sat", "supp")
FLUID_TOKENS = ("t2", "stir", "pd", "dess", "spair", "spir", "tirm")


def has_token(text: str, tokens) -> bool:
    t = text.lower().replace("-", "").replace(" ", "")
    return any(tok.replace("_", "") in t for tok in tokens)


def plane_from_iop(iop) -> str:
    if iop is None or len(iop) != 6:
        return "unknown"
    n = np.cross(np.array(iop[:3], float), np.array(iop[3:], float))
    return {0: "Sagittal", 1: "Coronal", 2: "Axial"}[int(np.argmax(np.abs(n)))]


def classify_weighting(tr, te, scanning_seq: str, desc: str) -> str:
    d = desc.lower()
    # Gradient echo has a short TR by design, so the TR/TE rule does not apply.
    if "gr" in scanning_seq.lower() or any(t in d for t in ("gre", "dess", "medic", "flash")):
        return "GRE"
    if tr is None or te is None:
        for k in ("t1", "t2", "pd"):
            if k in d:
                return k.upper()
        return "unknown"
    if tr <= TR_SHORT_MAX:
        return "T1"
    return "T2" if te >= TE_LONG_MIN else "PD"


def _f(v):
    try:
        return float(v)
    except Exception:
        return None


def centre_x_mm(h):
    """Patient-space x (LPS: +x = patient's left) of the image centre, in mm. The
    Laterality tag is missing on ~half the corpus; this is what decides the knee side."""
    ipp = getattr(h, "ImagePositionPatient", None)
    iop = getattr(h, "ImageOrientationPatient", None)
    ps = getattr(h, "PixelSpacing", None)
    rows, cols = getattr(h, "Rows", None), getattr(h, "Columns", None)
    if None in (ipp, iop, ps, rows, cols) or len(iop) != 6:
        return None
    r = np.array(iop[:3], float)          # direction of increasing column
    c = np.array(iop[3:], float)          # direction of increasing row
    centre = (np.array(ipp, float) + r * (float(cols) / 2) * float(ps[1])
              + c * (float(rows) / 2) * float(ps[0]))
    return float(centre[0])


def study_side(sdf, dead_zone_mm):
    """('L'|'R'|'', tag, geometry, conflict) for one study -- same rule as the cache."""
    tags = [t for t in sdf.get("laterality_tag", pd.Series(dtype=str)).tolist() if t in ("L", "R")]
    tag = max(set(tags), key=tags.count) if tags else ""
    xs = sdf["centre_x_mm"].dropna().to_numpy(dtype=float) if "centre_x_mm" in sdf else np.array([])
    geo = ""
    if len(xs):
        med = float(np.median(xs))
        if med > dead_zone_mm:
            geo = "L"
        elif med < -dead_zone_mm:
            geo = "R"
    conflict = int(bool(tag) and bool(geo) and tag != geo)
    side = "" if conflict else (tag if tag else geo)
    return side, tag, geo, conflict


def scan_series(series_csv: str, image_root: str, cache: str,
                max_studies: int = 0) -> pd.DataFrame:
    """One row per series with header-derived properties. Cached, because reading
    ~24k headers is slow and a resumed session must not pay for it twice."""
    if max_studies:                    # a smoke scan must never be mistaken for a full one
        cache = cache.replace(".csv", f"_smoke{max_studies}.csv")
    if os.path.exists(cache):
        print(f"  series cache hit: {cache}")
        return pd.read_csv(cache)

    meta = pd.read_csv(series_csv)
    if max_studies:
        keep = meta.StudyInstanceUID.drop_duplicates().head(max_studies)
        meta = meta[meta.StudyInstanceUID.isin(set(keep))]
        print(f"  smoke: scanning {len(meta)} series from {len(keep)} studies only")
    rows = []
    t0 = time.time()
    for i, r in enumerate(meta.itertuples(index=False)):
        d = os.path.join(image_root, r.StudyInstanceUID, r.SeriesInstanceUID)
        if not os.path.isdir(d):
            continue
        files = sorted(f for f in os.listdir(d) if f.endswith(".dcm"))
        if not files:
            # Do not assume the hidden test tree keeps the .dcm extension.
            files = sorted(f for f in os.listdir(d)
                           if os.path.isfile(os.path.join(d, f)))
        if not files:
            continue
        h = None
        for f in files[:5]:            # first file that parses, not blindly files[0]
            try:
                h = pydicom.dcmread(os.path.join(d, f), stop_before_pixels=True)
                break
            except Exception:
                continue
        if h is None:
            continue
        desc = " ".join(str(getattr(h, k, "") or "") for k in
                        ("SeriesDescription", "SequenceName", "ScanOptions", "ProtocolName"))
        trv = getattr(h, "RepetitionTime", None)
        tev = getattr(h, "EchoTime", None)
        w = classify_weighting(float(trv) if trv is not None else None,
                               float(tev) if tev is not None else None,
                               str(getattr(h, "ScanningSequence", "") or ""), desc)
        plane = getattr(r, "Anatomical_Plane", None)
        if not isinstance(plane, str) or plane not in ("Sagittal", "Coronal", "Axial"):
            plane = plane_from_iop(getattr(h, "ImageOrientationPatient", None))
        rows.append({
            "StudyInstanceUID": r.StudyInstanceUID,
            "SeriesInstanceUID": r.SeriesInstanceUID,
            "n_slices": len(files),
            "plane": plane,
            "weighting": w,
            "fat_sat": int(has_token(desc, FATSAT_TOKENS)),
            "fluid": int(w in ("T2", "PD") or has_token(desc, FLUID_TOKENS)),
            "laterality_tag": (str(getattr(h, "Laterality", "") or getattr(h, "ImageLaterality", "") or "").upper()
                               if str(getattr(h, "Laterality", "") or getattr(h, "ImageLaterality", "") or "").upper() in ("L", "R") else ""),
            "centre_x_mm": centre_x_mm(h),
        })
        if (i + 1) % 2000 == 0:
            print(f"    {i+1}/{len(meta)} series  {time.time()-t0:.0f}s")
    df = pd.DataFrame(rows)
    if len(df):
        df.to_csv(cache, index=False)
        print(f"  scanned {len(df)} series in {time.time()-t0:.0f}s -> {cache}")
    else:
        # Never cache an empty scan: a resumed session would hit the empty cache and
        # silently train on nothing.
        print(f"  scanned 0 series under {image_root} (cache NOT written)")
    return df


SLOT_SPEC = {
    "SAG_FLUID_FS":   ("Sagittal", lambda r: r.fluid and r.fat_sat, lambda r: r.fluid),
    "COR_FLUID_FS":   ("Coronal",  lambda r: r.fluid and r.fat_sat, lambda r: r.fluid),
    "AX_FLUID_FS":    ("Axial",    lambda r: r.fluid and r.fat_sat, lambda r: r.fluid),
    "SAG_FLUID_NOFS": ("Sagittal", lambda r: r.fluid and not r.fat_sat, lambda r: r.fluid),
    "COR_T1":         ("Coronal",  lambda r: r.weighting == "T1", lambda r: not r.fluid),
    "SAG_T1":         ("Sagittal", lambda r: r.weighting == "T1", lambda r: not r.fluid),
}


def select_slots(sdf: pd.DataFrame) -> dict:
    """One series per slot; strict tier across all slots first, then relaxed, so a
    series claimed strictly is not stolen by another slot's fallback. Prefers a
    slice count near 32 to avoid unusually long 3D / high-resolution acquisitions."""
    out, used = {}, set()
    for tier in (1, 2):
        for slot, (plane, strict, relaxed) in SLOT_SPEC.items():
            if slot in out:
                continue
            pred = strict if tier == 1 else relaxed
            cand = sdf[(sdf.plane == plane) & sdf.apply(pred, axis=1)]
            cand = cand[~cand.SeriesInstanceUID.isin(used)]
            if len(cand) == 0:
                continue
            chosen = cand.iloc[(cand.n_slices - 32).abs().to_numpy().argmin()]
            out[slot] = chosen.SeriesInstanceUID
            used.add(chosen.SeriesInstanceUID)
    return out


def build_manifest(series_df: pd.DataFrame, cache: str) -> pd.DataFrame:
    if os.path.exists(cache):
        print(f"  manifest cache hit: {cache}")
        return pd.read_csv(cache)
    rows = []
    for study, sdf in series_df.groupby("StudyInstanceUID"):
        slots = select_slots(sdf)
        side, tag, geo, conflict = study_side(sdf, cfg.lat_dead_zone_mm)
        rows.append({"StudyInstanceUID": study,
                     **{s: slots.get(s, "") for s in SLOTS},
                     "n_slots": len(slots), "side": side, "side_tag": tag,
                     "side_geo": geo, "side_conflict": conflict})
    m = pd.DataFrame(rows)
    m.to_csv(cache, index=False)
    print(f"  manifest -> {cache}; mean slots/study {m.n_slots.mean():.2f}; side resolved "
          f"{(m.side != '').mean():.1%} (tag {(m.side_tag != '').mean():.1%}, conflicts "
          f"{int(m.side_conflict.sum())})")
    print("  slot fill rate:",
          {s: round(float((m[s] != '').mean()), 3) for s in SLOTS})
    return m

## Section 4: reading pixels

Four things that produce **no error** if you get them wrong:

1. **Slice order.** The filename is the SOP Instance UID, assigned to be unique
   rather than ordered. Measured on the sample studies: Spearman ρ between
   filename order and true spatial position is **−0.012** on average, and
   `|ρ|>0.99` in **0 of 12** series. Sorting by filename silently destroys the
   slice adjacency that makes a 2.5D triplet meaningful. Sort by projecting
   `ImagePositionPatient` onto the slice normal from `ImageOrientationPatient`.
2. **Rescale and photometric.** Apply `RescaleSlope`/`Intercept`; invert
   `MONOCHROME1`. The sample studies happen to be all `MONOCHROME2` with trivial
   rescale, but the hidden test set spans 16–19 sites.
3. **Per-series normalisation.** Max intensity spans 690 … 8,736 across sample
   series (12.7×). A global window would not transfer. Clip each triplet jointly
   at its 1st/99th percentile so its three channels stay mutually comparable.
4. **Multi-frame files.** Some DICOMs hold a volume in one file; take the middle
   frame rather than crashing on the extra axis.

In [ ]:
# ── Section 4: pixels ─────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
GRAY_MEAN, GRAY_STD = 0.449, 0.226     # ImageNet mean/std averaged over RGB, for N-channel stacks


def ordered_slice_paths(series_dir: str, plane: str = None, return_head: bool = False):
    """Spatially ordered slice paths. NEVER trust filename order.

    With `plane` given (cache path) the sort direction has a FIXED sign: sagittal
    stacks run along +x (patient left), other planes along the positive dominant axis,
    so "reverse for right knees" canonicalises rather than randomises between sites.
    Without `plane` (legacy decode path) the cross-product normal is used as before.
    `return_head=True` also returns the header of the FIRST FILE IN FILENAME ORDER -- the one
    the cache builder reads IOP / PixelSpacing from (src/cache_pipeline.py::ordered_slice_paths);
    reading the spatially-first slice instead was a latent divergence between the two."""
    files = [f for f in os.listdir(series_dir) if f.endswith(".dcm")]
    if not files:   # do not assume the hidden test tree keeps the .dcm extension
        files = [f for f in os.listdir(series_dir)
                 if os.path.isfile(os.path.join(series_dir, f))]
    if not files:
        return ([], None) if return_head else []
    paths = [os.path.join(series_dir, f) for f in sorted(files)]
    heads, kept = [], []
    for p in paths:
        try:
            heads.append(pydicom.dcmread(p, stop_before_pixels=True))
            kept.append(p)
        except Exception:
            continue                    # a stray non-DICOM file must not poison the order
    paths = kept
    if not heads:
        return ([], None) if return_head else []
    first = heads[0]

    def done(ordered):
        return (ordered, first) if return_head else ordered

    iop = getattr(first, "ImageOrientationPatient", None)
    if iop is not None and len(iop) == 6:
        n = np.cross(np.array(iop[:3], float), np.array(iop[3:], float))
        if plane == "Sagittal":
            n = np.array([1.0, 0.0, 0.0])
        elif plane is not None and n[int(np.argmax(np.abs(n)))] < 0:
            n = -n
        keys, ok = [], True
        for h in heads:
            ipp = getattr(h, "ImagePositionPatient", None)
            if ipp is None:
                ok = False
                break
            keys.append(float(np.dot(np.array(ipp, float), n)))
        if ok:
            return done([p for _, p in sorted(zip(keys, paths), key=lambda t: t[0])])
    inst = [getattr(h, "InstanceNumber", None) for h in heads]
    if all(i is not None for i in inst):
        return done([p for _, p in sorted(zip(inst, paths), key=lambda t: t[0])])
    print(f"  ! {series_dir}: no usable position/instance headers -- filename order")
    return done(paths)


def read_plane(path: str) -> np.ndarray:
    ds = pydicom.dcmread(path)
    arr = ds.pixel_array.astype(np.float32)
    if arr.ndim == 3:                      # multi-frame: middle frame
        arr = arr[arr.shape[0] // 2]
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    inter = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + inter
    if str(getattr(ds, "PhotometricInterpretation", "")).upper() == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def build_triplets(series_dir: str, n_samples: int, gap: int, size: int) -> torch.Tensor:
    """-> (n_samples, 3, size, size). Channels are slices [i-gap, i, i+gap], so the
    encoder sees local 3D context through a 2D backbone."""
    ordered = ordered_slice_paths(series_dir)
    if not ordered:
        return torch.zeros(n_samples, 3, size, size)
    n = len(ordered)
    centres = np.clip(np.linspace(gap, n - 1 - gap, n_samples).round().astype(int), 0, n - 1)
    out = []
    for c in centres:
        idx = [max(0, c - gap), int(c), min(n - 1, c + gap)]
        try:
            planes = [read_plane(ordered[i]) for i in idx]
        except Exception:
            out.append(torch.zeros(3, size, size))
            continue
        h = min(p.shape[0] for p in planes)
        w = min(p.shape[1] for p in planes)
        stack = np.stack([p[:h, :w] for p in planes], axis=0).astype(np.float32)
        lo, hi = np.percentile(stack, [1, 99])     # joint clip keeps channels comparable
        stack = (np.clip(stack, lo, hi) - lo) / max(hi - lo, 1e-6)
        t = torch.from_numpy(stack).unsqueeze(0)
        t = F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)
        t = (t.squeeze(0) - IMAGENET_MEAN) / IMAGENET_STD
        out.append(t)
    return torch.stack(out)

def centre_crop_mm(arr, pixel_spacing, crop_mm):
    if not crop_mm or pixel_spacing is None or pixel_spacing <= 0:
        return arr
    side_px = int(round(crop_mm / pixel_spacing))
    h, w = arr.shape
    if side_px >= min(h, w):
        return arr
    y0 = (h - side_px) // 2
    x0 = (w - side_px) // 2
    return arr[y0:y0 + side_px, x0:x0 + side_px]


def resize_u8(stack01, px):
    t = torch.from_numpy(np.ascontiguousarray(stack01)).unsqueeze(1)
    t = F.interpolate(t, size=(px, px), mode="bilinear", align_corners=False)
    return (t.squeeze(1).clamp_(0, 1) * 255).round().to(torch.uint8).numpy()


def cache_series(series_dir, plane, cfg, is_right, n_slices, band=None, px=None):
    """-> ((n_slices, px, px) uint8, n_failed) or (None, n_failed).
    IDENTICAL to src/cache_pipeline.py::cache_series -- keep them in sync (src/cache_selftest.py
    checks both schemes bit for bit). Used at test time so a test study gets exactly the
    preprocessing the cached training studies got. `band` is the plane's (lo, hi) fraction of
    the ordered stack and `px` the stored resolution; both default to the c01 values."""
    ordered, head = ordered_slice_paths(series_dir, plane, return_head=True)
    if not ordered:
        return None, 0
    n = len(ordered)
    lo_f, hi_f = band if band is not None else CACHE_BAND.get(plane, (0.0, 1.0))
    lo_i, hi_i = int(round(lo_f * (n - 1))), int(round(hi_f * (n - 1)))
    if hi_i <= lo_i:
        lo_i, hi_i = 0, n - 1
    # Repeated neighbours on short series are intended (no np.unique).
    idx = np.linspace(lo_i, hi_i, n_slices).round().astype(int)
    if plane == "Sagittal" and is_right:
        idx = idx[::-1]
    iop = getattr(head, "ImageOrientationPatient", None)
    col_to_left = (iop is not None and len(iop) == 6 and float(iop[0]) > 0)
    mirror = plane in ("Coronal", "Axial") and (col_to_left == is_right)
    ps = getattr(head, "PixelSpacing", None)
    ps = float(ps[0]) if ps is not None else None
    planes, n_fail = [], 0
    for i in idx:
        try:
            a = read_plane(ordered[int(i)])
        except Exception:
            a = None
            n_fail += 1
        planes.append(a)
    good = [a for a in planes if a is not None]
    if not good:
        return None, n_fail
    h = min(a.shape[0] for a in good)
    w = min(a.shape[1] for a in good)
    # A failed slice is replaced by its nearest good neighbour, never by zeros (zeros
    # would drag the per-series percentiles down and enter the model as a black slice).
    fixed = []
    for k, a in enumerate(planes):
        if a is None:
            near = min((j for j, b in enumerate(planes) if b is not None), key=lambda j: abs(j - k))
            a = planes[near]
        fixed.append(a[:h, :w])
    stack = np.stack(fixed).astype(np.float32)
    stack = np.stack([centre_crop_mm(x, ps, cfg.crop_mm) for x in stack])
    lo, hi = np.percentile(stack, [CACHE_PCT[0], CACHE_PCT[1]])   # per SERIES, whole stack
    stack = (np.clip(stack, lo, hi) - lo) / max(hi - lo, 1e-6)
    if mirror:
        stack = stack[:, :, ::-1]
    return resize_u8(stack, px if px is not None else cfg.cache_px), n_fail


def build_study_array(study, row, image_root, cfg):
    """On-the-fly equivalent of one cached study, in the layout of `cfg`'s cache scheme:
    c01 -> ([6, S, P, P] uint8, mask[6]); c02 -> ([sum(budgets), P, P] uint8, mask[6]) with slot
    `si` at rows slot_offsets()[si]. Mirrors cache_study / build_study_flat in the builder."""
    scheme, px, slot_slices, band = cache_geom(cfg)
    starts, total = slot_offsets(slot_slices)
    if scheme == "c01":
        arr = np.zeros((len(SLOTS), slot_slices[0], px, px), np.uint8)
    else:
        arr = np.zeros((total, px, px), np.uint8)
    mask = np.zeros(len(SLOTS), np.float32)
    is_right = str(row.get("side", "")) == "R"
    for si, slot in enumerate(SLOTS):
        sid = row[slot]
        if not isinstance(sid, str) or not sid:
            continue
        d = os.path.join(image_root, study, sid)
        if not os.path.isdir(d):
            continue
        plane = PLANE_OF_SLOT[slot]
        a, _ = cache_series(d, plane, cfg, is_right, slot_slices[si], band=band[plane], px=px)
        if a is None:
            continue
        if scheme == "c01":
            arr[si] = a
        else:
            arr[starts[si]:starts[si] + slot_slices[si]] = a
        mask[si] = 1.0
    return arr, mask


def slot_stacks(arr, cfg):
    """The six per-slot (n_i, P, P) views of a cached study, for either layout: c01 arrays are
    [6, S, P, P] (view = arr[si]); c02 arrays are flat [sum, P, P] (view = a row range)."""
    if arr.ndim == 4:
        return [arr[si] for si in range(len(SLOTS))]
    _, _, slot_slices, _ = cache_geom(cfg)
    starts, _ = slot_offsets(slot_slices)
    return [arr[s:s + n] for s, n in zip(starts, slot_slices)]


_NPY_HEADERS = {}     # blob path -> (shape, dtype, header_bytes); per process (DataLoader worker)


def npy_header(path):
    """(shape, dtype, header_bytes) of a .npy file, public numpy API only."""
    with open(path, "rb") as f:
        version = np.lib.format.read_magic(f)
        reader = {(1, 0): np.lib.format.read_array_header_1_0,
                  (2, 0): np.lib.format.read_array_header_2_0}.get(version)
        if reader is None:
            raise ValueError(f"unsupported .npy version {version} in {path}")
        shape, fortran, dtype = reader(f)
        if fortran:
            raise ValueError(f"{path} is Fortran-ordered; blobs must be C-ordered")
        return tuple(shape), dtype, f.tell()


def read_cached(locator):
    """One study's uint8 array from its locator: a .npy path (c01) or (blob_path, row) (c02).
    The blob read is a single seek + read of that study's bytes -- no np.load(mmap_mode) on
    Kaggle's FUSE input mount, no mapping held open inside DataLoader workers, and the 8 MB
    buffer is freed with the item (the design review's memory concern, 2026-08-30)."""
    if isinstance(locator, str):
        return np.load(locator)
    path, row = locator
    hdr = _NPY_HEADERS.get(path)
    if hdr is None:
        hdr = _NPY_HEADERS[path] = npy_header(path)
    shape, dtype, header_bytes = hdr
    if not (0 <= row < shape[0]):
        raise IndexError(f"row {row} outside blob {path} with {shape[0]} studies")
    per_study = int(np.prod(shape[1:]))
    itemsize = np.dtype(dtype).itemsize
    with open(path, "rb") as f:
        f.seek(header_bytes + row * per_study * itemsize)
        buf = np.fromfile(f, dtype=dtype, count=per_study)
    if buf.size != per_study:
        raise IOError(f"short read on {path} row {row}: {buf.size} of {per_study} elements")
    return buf.reshape(shape[1:])


def valid_windows(mask, cfg):
    """Every (slot, centre) triplet window a study offers: centres 1 .. n_i-2 of each PRESENT
    slot. Returns (centres, slot_id) as int arrays; the centre indexes the slot's own stack."""
    _, _, slot_slices, _ = cache_geom(cfg)
    cs, ss = [], []
    for si, n in enumerate(slot_slices):
        if float(mask[si]) <= 0:
            continue
        c = np.arange(1, int(n) - 1)
        cs.append(c)
        ss.append(np.full(len(c), si, dtype=np.int64))
    if not cs:
        return np.zeros(0, np.int64), np.zeros(0, np.int64)
    return np.concatenate(cs), np.concatenate(ss)


def sample_train_windows(centres, slot_id, n, min_per_slot=2):
    """Training view: `n` windows without replacement, stratified so every present slot keeps at
    least `min_per_slot` (if it has that many), the rest uniform over what is left. Uses the
    global numpy RNG, which seed_worker re-seeds per worker and epoch."""
    W = len(centres)
    if n >= W:
        order = np.random.permutation(W)          # every window, shuffled
        return centres[order], slot_id[order]
    chosen = []
    for si in np.unique(slot_id):
        pool = np.flatnonzero(slot_id == si)
        k = min(min_per_slot, len(pool), max(0, n - len(chosen)))
        if k:
            chosen.extend(np.random.choice(pool, k, replace=False).tolist())
    rest = np.setdiff1d(np.arange(W), np.array(chosen, dtype=np.int64))
    need = n - len(chosen)
    if need > 0:
        chosen.extend(np.random.choice(rest, need, replace=False).tolist())
    ix = np.array(sorted(chosen), dtype=np.int64)
    return centres[ix], slot_id[ix]


def eval_windows_subset(centres, slot_id, n_eval):
    """Evaluation view: all windows when n_eval <= 0 or >= W; otherwise n_eval windows spread
    equidistantly over the (slot-ordered) list -- the same rule for oof_eval and infer."""
    W = len(centres)
    if n_eval <= 0 or n_eval >= W:
        return centres, slot_id
    ix = np.linspace(0, W - 1, n_eval).round().astype(np.int64)
    return centres[ix], slot_id[ix]


def array_to_tensor(arr, mask, cfg, train, centre_offset=0):
    """[6, S, P, P] uint8 -> (6, K, 3, img, img) float normalised for the encoder.
    Triplet channels are neighbouring cached slices [c-1, c, c+1]; the K centres are
    equidistant over the interior of the stack (eval) -- the same for train in v03 so the
    cache experiment isolates the cache, not a new augmentation. `centre_offset` shifts every
    centre by that many cached slices (clipped) -- the slice-offset TTA views (P-12); 0 is
    bit-identical to the pre-TTA code. A flat c02 array is handled slot by slot (ragged S)."""
    if arr.ndim == 3:                                   # c02 flat layout: per-slot stacks
        K = cfg.slices_per_slot
        views = []
        for st in slot_stacks(arr, cfg):
            S = st.shape[0]
            centres = np.linspace(1, S - 2, K).round().astype(int)
            if train and getattr(cfg, "cache_jitter", False):
                centres = centres + np.random.randint(-1, 2, size=K)
            centres = np.clip(centres + centre_offset, 1, S - 2)
            idx = np.stack([centres - 1, centres, centres + 1], axis=1)
            views.append(torch.from_numpy(st[idx].astype(np.float32) / 255.0))   # (K, 3, P, P)
        x = torch.stack(views)                                                    # (6, K, 3, P, P)
        if x.shape[-1] != cfg.img_size:
            x = F.interpolate(x.reshape(-1, 3, x.shape[-2], x.shape[-1]),
                              size=(cfg.img_size, cfg.img_size), mode="bilinear",
                              align_corners=False).reshape(len(SLOTS), K, 3, cfg.img_size, cfg.img_size)
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
        m = torch.as_tensor(np.asarray(mask, dtype=np.float32))
        return x * m.view(-1, 1, 1, 1, 1), m
    S = arr.shape[1]
    if getattr(cfg, "stack_mode", "triplet") == "channels":
        idx = np.arange(S)
        if train and getattr(cfg, "cache_jitter", False):
            idx = np.clip(idx + np.random.randint(-1, 2), 0, S - 1)   # shift the stack +-1 slice
        x = torch.from_numpy(arr[:, idx].astype(np.float32) / 255.0).unsqueeze(1)   # (6, 1, S, P, P)
        if x.shape[-1] != cfg.img_size:
            x = F.interpolate(x.reshape(-1, S, x.shape[-2], x.shape[-1]),
                              size=(cfg.img_size, cfg.img_size), mode="bilinear",
                              align_corners=False).reshape(len(SLOTS), 1, S, cfg.img_size, cfg.img_size)
        x = (x - GRAY_MEAN) / GRAY_STD
        m = torch.as_tensor(np.asarray(mask, dtype=np.float32))
        return x * m.view(-1, 1, 1, 1, 1), m
    K = cfg.slices_per_slot
    centres = np.linspace(1, S - 2, K).round().astype(int)
    if train and getattr(cfg, "cache_jitter", False):
        centres = np.clip(centres + np.random.randint(-1, 2, size=K), 1, S - 2)
    if centre_offset:
        centres = np.clip(centres + centre_offset, 1, S - 2)
    idx = np.stack([centres - 1, centres, centres + 1], axis=1)          # (K, 3)
    x = torch.from_numpy(arr[:, idx].astype(np.float32) / 255.0)         # (6, K, 3, P, P)
    if x.shape[-1] != cfg.img_size:
        x = F.interpolate(x.reshape(-1, 3, x.shape[-2], x.shape[-1]),
                          size=(cfg.img_size, cfg.img_size), mode="bilinear",
                          align_corners=False).reshape(len(SLOTS), K, 3, cfg.img_size, cfg.img_size)
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    m = torch.as_tensor(np.asarray(mask, dtype=np.float32))
    x = x * m.view(-1, 1, 1, 1, 1)                # absent slots stay exactly zero
    return x, m


def undo_laterality(arr, cfg):
    """P-05 ablation: put a right knee back into its own chirality.

    The cache stores every study in a canonical left-knee frame -- coronal/axial mirrored
    left-right, sagittal stacks reversed. Both are involutions, so re-applying them to the
    R studies restores the two-chirality condition P-05 removed, with no cache rebuild.

    It does not reconstruct the original bytes: the per-series `col_to_left` sign that
    decided the mirror is not in the manifest. It reproduces the thing being ablated --
    chirality that varies with knee side -- which is what the arm is asking about. This is
    a cleaner test than v03-vs-v02, where the 130 mm crop varied at the same time.
    """
    out = arr.copy()
    for si, (slot, st) in enumerate(zip(SLOTS, slot_stacks(out, cfg))):
        if PLANE_OF_SLOT[slot] == "Sagittal":
            st[:] = st[::-1].copy()           # reverse the slice axis
        else:
            st[:] = st[:, :, ::-1].copy()     # mirror the width axis (coronal / axial)
    return np.ascontiguousarray(out)

## Section 5: dataset

One item = one study: a `(slot, slices, 3, H, W)` tensor plus a presence mask.
Absent slots are zero-filled and masked, which is why the head receives the mask
explicitly — "this study had no axial fluid series" is information, not noise.

**Laterality normalisation:** right knees are mirrored so medial/lateral means the
same thing in every image. Without it the model has to learn each finding twice,
and `Medial OA` vs `Lateral OA` are separate labels — mirroring is not cosmetic.
The DICOM tag is unreliable in this corpus, so this uses a light heuristic and
leaves a hook for a better one.

In [ ]:
# ── Section 5: dataset ────────────────────────────────────────────────────────
class KneeStudyDataset(Dataset):
    def __init__(self, manifest, targets_df, image_root, cfg, train=True,
                 studies=None):
        self.m = manifest.set_index("StudyInstanceUID")
        self.t = targets_df.set_index("StudyInstanceUID") if targets_df is not None else None
        self.root = image_root
        self.cfg = cfg
        self.train = train
        keep = studies if studies is not None else list(self.m.index)
        self.studies = [s for s in keep if s in self.m.index]

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, i):
        study = self.studies[i]
        row = self.m.loc[study]
        if self.cfg.use_cache:
            locator = CACHE_INDEX.get(cache_version_for(self.cfg), {}).get(study)
            if locator is not None:
                arr = read_cached(locator)
                mk = str(row["mask"]) if "mask" in row and isinstance(row["mask"], str) else None
                if mk is None or len(mk) != len(SLOTS):
                    mk = "".join("1" if st.any() else "0" for st in slot_stacks(arr, self.cfg))
                mask_np = np.array([float(c) for c in mk], np.float32)
            else:                       # test study, or a study the cache missed
                arr, mask_np = build_study_array(study, row, self.root, self.cfg)
            if self.cfg.lat_undo and str(row.get("side", "")) == "R":
                arr = undo_laterality(arr, self.cfg)   # P-05 ablation arm; counted in train_fold
            if getattr(self.cfg, "window_mode", "fixed") == "random":
                # P-25: ship the uint8 study + window indices; the model gathers, normalises and
                # resizes on the GPU (60 float windows per study would otherwise cross the
                # DataLoader shared-memory boundary at ~80-100 MB each).
                centres, slot_id = valid_windows(mask_np, self.cfg)
                if self.train:
                    centres, slot_id = sample_train_windows(centres, slot_id, self.cfg.train_windows)
                else:
                    centres, slot_id = eval_windows_subset(centres, slot_id, self.cfg.eval_windows)
                out = {"study": study, "arr": torch.from_numpy(np.ascontiguousarray(arr)),
                       "centres": torch.from_numpy(centres.astype(np.int64)),
                       "slot_id": torch.from_numpy(slot_id.astype(np.int64)),
                       "mask": torch.as_tensor(mask_np)}
                if self.t is not None:
                    r = self.t.loc[study]
                    out["y"] = torch.tensor([float(r[l]) for l in LABELS])
                    out["w"] = torch.tensor([float(r[f"w__{l}"]) for l in LABELS])
                    out["is_gold"] = torch.tensor(float(r["is_gold"]))
                return out
            offsets = (0,) if self.train else tuple(getattr(self.cfg, "tta_offsets", (0,)))
            views = [array_to_tensor(arr, mask_np, self.cfg, self.train, centre_offset=o)
                     for o in offsets]
            imgs, mask = views[0]
            if len(views) > 1:
                imgs = torch.stack([v[0] for v in views])        # (n_views, 6, K, 3, H, W)
        else:
            imgs = torch.zeros(len(SLOTS), self.cfg.slices_per_slot, 3,
                               self.cfg.img_size, self.cfg.img_size)
            mask = torch.zeros(len(SLOTS))
            for si, slot in enumerate(SLOTS):
                sid = row[slot]
                if not isinstance(sid, str) or not sid:
                    continue
                d = os.path.join(self.root, study, sid)
                if not os.path.isdir(d):
                    continue
                imgs[si] = build_triplets(d, self.cfg.slices_per_slot,
                                          self.cfg.triplet_gap, self.cfg.img_size)
                mask[si] = 1.0

        if self.train:
            # Light augmentation. No vertical flip: knee anatomy is not
            # up/down symmetric, and no horizontal flip either because that
            # would swap medial and lateral -- which are different labels.
            if random.random() < 0.5:
                imgs = imgs + torch.randn_like(imgs) * 0.01

        out = {"study": study, "imgs": imgs, "mask": mask}
        if self.t is not None:
            r = self.t.loc[study]
            out["y"] = torch.tensor([float(r[l]) for l in LABELS])
            out["w"] = torch.tensor([float(r[f"w__{l}"]) for l in LABELS])
            out["is_gold"] = torch.tensor(float(r["is_gold"]))
        return out

## Section 6: model

```
study -> 6 slots -> N triplets each
                      |
            shared DINOv2 ViT-S/14  (one encoder for all slots: 4,407 studies
                      |              cannot support six separate encoders)
         attention pool over slices  (a torn ACL is visible on a few slices, so
                      |               mean pooling dilutes it ~6x)
           concat 6 slot vectors + 6-bit presence mask
                      |
                 linear -> 12 logits
```

Two rates: the head gets `lr_head` (1e-3); the backbone gets `lr_backbone`
(2e-5) at its top block, decaying by 0.75 per block downwards (layer-wise LR
decay), and an EMA of the weights is what gets validated and saved. The
pretrained self-supervised features are the asset here — with 58 gold labels
there is nowhere near enough signal to relearn them, so they are nudged, not
retrained. Every medical DINOv2 recipe we found sits at 1e-6..2e-5; a uniform
5e-5 (v01) is the "catastrophic forgetting" regime — see docs/research.md.

In [ ]:
# ── Section 6: model ──────────────────────────────────────────────────────────
class AttnPool(nn.Module):
    """Attention pooling over the slice axis.

    Mean pooling weights every slice equally, so a finding visible on 1 of 6
    sampled slices is diluted. This learns which slices matter.
    """

    def __init__(self, dim: int):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(dim, dim // 4), nn.Tanh(),
                                   nn.Linear(dim // 4, 1))

    def forward(self, x):                    # x: (S, dim)
        a = torch.softmax(self.score(x).squeeze(-1), dim=0)
        return (a.unsqueeze(-1) * x).sum(0)


class SlotAttnHead(nn.Module):
    """P-09: 12 learned label queries attending over the slot vectors that are present.

    The concat head maps [6 x dim | mask] through one Linear, so every label reads all six
    slots through one shared weight matrix: "for MCL, weight coronal and ignore axial" has
    to be learned as 12 independent 2,310-dim rows from 3,525 studies of noisy targets.
    Here each label owns a query, a per-(label, slot) bias states that plane preference in
    72 parameters, and absent slots are masked out *before* the softmax so the context
    vector has the same scale whether a study has four slots or six (mean slots is 4.78 of
    6; COR_T1 fills 62.5%, SAG_T1 50%). 9,300 parameters against the concat head's 27,720.

    Risk on record (research.md): correlated label pairs may lose the shared-vector
    benefit -- report Effusion~Synovitis, Medial OA~Medial Meniscus and Contusion~Fracture
    separately, not just the macro.
    """

    def __init__(self, dim: int, n_labels=len(LABELS), n_slots=len(SLOTS)):
        super().__init__()
        self.q = nn.Parameter(torch.randn(n_labels, dim) * dim ** -0.5)
        # 2-D, so param_groups gives it weight decay. Decaying it toward zero is a
        # uniform-plane prior, which is the right default for a term with no data yet.
        self.slot_bias = nn.Parameter(torch.zeros(n_labels, n_slots))
        self.w = nn.Parameter(torch.randn(n_labels, dim) * dim ** -0.5)
        self.b = nn.Parameter(torch.zeros(n_labels))
        self.scale = dim ** -0.5

    def forward(self, pooled, mask):             # pooled (B, NS, dim), mask (B, NS)
        att = torch.einsum("ld,bsd->bls", self.q, pooled) * self.scale
        att = att + self.slot_bias.unsqueeze(0)
        keep = (mask > 0.5).unsqueeze(1)                             # (B, 1, NS)
        att = att.masked_fill(~keep, torch.finfo(att.dtype).min)     # fp16-safe, not -inf
        # A study with no present slot cannot reach here (the manifest requires
        # n_slots > 0), but an all-masked row would softmax to NaN. Fall back to uniform.
        dead = (~keep).all(-1, keepdim=True).expand_as(att)
        att = torch.where(dead, torch.zeros_like(att), att)
        ctx = torch.einsum("bls,bsd->bld", torch.softmax(att, dim=-1), pooled)
        return (ctx * self.w.unsqueeze(0)).sum(-1) + self.b


def widen_patch_embedding(enc, in_chans):
    """3 -> `in_chans` input channels on a HF vision encoder (P-23 #3, stack_mode="channels").

    The pretrained RGB kernel is averaged over its three channels, replicated `in_chans` times and
    scaled by 3/in_chans, so a stack of identical slices produces exactly the response the grey
    image would have -- the model starts as "mean over the stack" and learns which slice offsets
    matter. Every `num_channels` bookkeeping attribute is updated because HF embeddings assert on
    it at forward time (Dinov2PatchEmbeddings, ConvNextEmbeddings)."""
    emb = enc.embeddings
    name, conv = next((n, m) for n, m in emb.named_modules() if isinstance(m, nn.Conv2d))
    new = nn.Conv2d(in_chans, conv.out_channels, conv.kernel_size, conv.stride,
                    conv.padding, bias=conv.bias is not None)
    with torch.no_grad():
        new.weight.copy_(conv.weight.mean(1, keepdim=True).repeat(1, in_chans, 1, 1)
                         * (3.0 / in_chans))
        if conv.bias is not None:
            new.bias.copy_(conv.bias)
    parent, parts = emb, name.split(".")
    for part in parts[:-1]:
        parent = getattr(parent, part)
    setattr(parent, parts[-1], new)
    for mod in (emb, getattr(emb, "patch_embeddings", None), enc.config):
        if mod is not None and hasattr(mod, "num_channels"):
            mod.num_channels = in_chans
    print(f"  patch embedding widened 3 -> {in_chans} channels (embeddings.{name})")


class WindowAttnHead(nn.Module):
    """P-25: 12 label queries over EVERY (slot, window) token of a study.

    The existing heads pool each slot's windows with a label-AGNOSTIC AttnPool first, so a
    Fracture slice and a meniscus slice in the same sagittal stack compete for one 384-d slot
    vector before any label reads it. Here each label runs its own softmax over all windows
    of the study (the 0.936 notebook's strongest member pools this way), with a learned slot
    embedding added to every token so "which sequence" survives the flattening. Gate =
    Linear(dim,256) -> Tanh -> Dropout -> Linear(256, 12); output = per-label context dot a
    per-label weight. Padded / absent windows are masked with finfo.min before the softmax
    (fp16-safe); an all-masked row falls back to uniform rather than NaN."""

    def __init__(self, dim, n_labels=len(LABELS), n_slots=len(SLOTS), slot_embed=True,
                 dropout=0.2, hidden=256):
        super().__init__()
        self.slot_emb = nn.Parameter(torch.zeros(n_slots, dim)) if slot_embed else None
        self.norm = nn.LayerNorm(dim)
        self.gate = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Dropout(dropout),
                                  nn.Linear(hidden, n_labels))
        self.w = nn.Parameter(torch.randn(n_labels, dim) * dim ** -0.5)
        self.b = nn.Parameter(torch.zeros(n_labels))

    def forward(self, feats, slot_id, valid=None):
        # feats (B, W, dim)   slot_id (B, W) long   valid (B, W) bool or None
        h = feats
        if self.slot_emb is not None:
            h = h + self.slot_emb[slot_id]
        h = self.norm(h)
        att = self.gate(h).transpose(1, 2)                       # (B, L, W)
        if valid is not None:
            keep = valid.unsqueeze(1)                            # (B, 1, W)
            att = att.masked_fill(~keep, torch.finfo(att.dtype).min)
            dead = (~keep).all(-1, keepdim=True).expand_as(att)
            att = torch.where(dead, torch.zeros_like(att), att)
        a = torch.softmax(att.float(), dim=-1).to(h.dtype)       # per-label softmax over windows
        ctx = torch.einsum("blw,bwd->bld", a, h)                 # (B, L, dim)
        return (ctx * self.w.unsqueeze(0)).sum(-1) + self.b


def affine_theta(rot_deg, zoom, dx, dy):
    """(N,) tensors -> (N, 2, 3) theta for F.affine_grid (output -> input coordinates, align_corners=False).

    zoom z > 1 zooms IN: the grid samples a source patch 1/z the size of the input, so the scale entries
    are 1/z (a scale of z would zoom out and pad). dx / dy are the shift as a fraction of the width /
    height; normalised coordinates span 2, so a 5 % shift is 0.10. Built in fp32 so it never meets
    autocast's fp16 (affine_grid raises on a dtype mismatch)."""
    rot = torch.deg2rad(rot_deg.float())
    c, s = torch.cos(rot), torch.sin(rot)
    inv = 1.0 / zoom.float()
    return torch.stack([torch.stack([c * inv, -s * inv, 2.0 * dx.float()], -1),
                        torch.stack([s * inv, c * inv, 2.0 * dy.float()], -1)], 1)


def augment_light(x, p=0.8):
    """P-33: per-window train-time augmentation of gathered windows. x (W, C, H, W) floats in [0, 1], any
    float dtype; returns the same dtype and shape. Each window is augmented with probability p: an affine
    warp (rotation U(-8, 8) deg, zoom-in U(1.00, 1.08), shift U(-5, 5) %, zero padding -- MRI background
    is black), then gamma U(0.8, 1.25) and gain U(0.9, 1.1), clamped to [0, 1]. No flips (P-05: medial and
    lateral are different labels). Draws torch's global RNG, so seed_all() reproduces it; p = 0 returns x."""
    n_win = x.shape[0]
    if n_win == 0 or p <= 0:
        return x
    pick = torch.rand(n_win, device=x.device) < p
    if not bool(pick.any()):
        return x
    n = int(pick.sum())
    dev = x.device
    with torch.autocast(device_type="cuda" if dev.type == "cuda" else "cpu", enabled=False):
        xs = x[pick].float()
        rot = (torch.rand(n, device=dev) * 2 - 1) * 8.0
        zoom = 1.0 + torch.rand(n, device=dev) * 0.08
        dx = (torch.rand(n, device=dev) * 2 - 1) * 0.05
        dy = (torch.rand(n, device=dev) * 2 - 1) * 0.05
        grid = F.affine_grid(affine_theta(rot, zoom, dx, dy), list(xs.shape), align_corners=False)
        xs = F.grid_sample(xs, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
        gamma = 0.8 + torch.rand(n, 1, 1, 1, device=dev) * 0.45
        gain = 0.9 + torch.rand(n, 1, 1, 1, device=dev) * 0.2
        xs = (xs.clamp_min(0.0) ** gamma * gain).clamp(0.0, 1.0)
    out = x.clone()
    out[pick] = xs.to(x.dtype)
    return out


def load_timm_backbone(arch, backbone_dir, grad_checkpoint=False):
    """timm model built offline from <backbone_dir>/model.safetensors (the HF timm repo files,
    mounted as a Kaggle Dataset). Loads strictly except for the classifier head, and REFUSES a
    silent architecture mismatch -- `strict=False` alone would happily train from scratch."""
    import timm
    from safetensors.torch import load_file
    enc = timm.create_model(arch, pretrained=False, num_classes=0)
    sd = load_file(os.path.join(backbone_dir, "model.safetensors"))
    head_keys = [k for k in sd if k.startswith("head.fc")]        # ImageNet classifier
    for k in head_keys:
        sd.pop(k)
    res = enc.load_state_dict(sd, strict=False)
    bad_unexpected = [k for k in res.unexpected_keys if not k.startswith("head.")]
    if res.missing_keys or bad_unexpected:
        raise SystemExit(f"timm {arch}: weights do not match the architecture -- missing "
                         f"{res.missing_keys[:5]} ({len(res.missing_keys)}), unexpected "
                         f"{bad_unexpected[:5]} ({len(bad_unexpected)})")
    print(f"  timm {arch}: loaded {len(sd)} tensors from {backbone_dir} (dropped head "
          f"{len(head_keys)}); num_features {enc.num_features}, {len(enc.stages)} stages, "
          f"grad_checkpoint={grad_checkpoint}")
    if grad_checkpoint and hasattr(enc, "set_grad_checkpointing"):
        enc.set_grad_checkpointing(True)
    return enc


class KneeNet(nn.Module):
    def __init__(self, backbone_dir: str, n_labels=len(LABELS), dropout=0.1,
                 head_type="concat", slot_dropout=0.0, backbone="dinov2", in_chans=3,
                 slot_embed=True, grad_checkpoint=False, img_size=224, aug="none"):
        super().__init__()
        self.backbone = backbone
        self.in_chans = in_chans
        self.img_size = img_size
        self.aug = aug                    # P-33: train-time only, applied inside forward_windows
        if backbone == "convnext_tiny":
            from transformers import ConvNextModel
            self.enc = ConvNextModel.from_pretrained(backbone_dir)
            self.dim = self.enc.config.hidden_sizes[-1]          # 768 for Tiny
        elif str(backbone).startswith("timm:"):
            self.enc = load_timm_backbone(backbone.split(":", 1)[1], backbone_dir, grad_checkpoint)
            self.dim = self.enc.num_features
        else:
            from transformers import Dinov2Model
            self.enc = Dinov2Model.from_pretrained(backbone_dir)
            self.dim = self.enc.config.hidden_size
        if in_chans != 3:
            widen_patch_embedding(self.enc, in_chans)
        self.drop = nn.Dropout(dropout)
        self.head_type = head_type
        self.slot_dropout = slot_dropout
        if head_type == "window_attn":
            self.window_head = WindowAttnHead(self.dim, n_labels, slot_embed=slot_embed)
        else:
            self.pool = AttnPool(self.dim)
            if head_type == "attn":
                self.attn_head = SlotAttnHead(self.dim, n_labels)
            else:
                self.head = nn.Linear(self.dim * len(SLOTS) + len(SLOTS), n_labels)

    def encode(self, x):
        """(N, C, H, W) normalised images -> (N, dim) one vector per image."""
        if str(self.backbone).startswith("timm:"):
            return self.enc(x)                               # num_classes=0 -> pooled features
        out = self.enc(pixel_values=x)
        if self.backbone == "convnext_tiny":
            return out.pooler_output                         # LayerNorm(global-avg-pool), (N, 768)
        return out.last_hidden_state[:, 0]                   # CLS token, (N, 384)

    def forward(self, imgs, mask):
        # imgs: (B, SLOT, S, C, H, W)   mask: (B, SLOT)   C = 3 (triplet) or 16 (channels, S = 1)
        B, NS, S = imgs.shape[0], imgs.shape[1], imgs.shape[2]
        flat = imgs.reshape(B * NS * S, *imgs.shape[3:])
        feats = self.encode(flat).reshape(B, NS, S, self.dim)
        if self.head_type == "window_attn":
            # fixed-window input through the window head: every (slot, centre) is a token,
            # tokens of absent slots are masked out
            slot_id = torch.arange(NS, device=feats.device).repeat_interleave(S).unsqueeze(0).expand(B, -1)
            valid = (mask > 0.5).repeat_interleave(S, dim=1)
            return self.window_head(self.drop(feats.reshape(B, NS * S, self.dim)), slot_id, valid)
        pooled = torch.stack([
            torch.stack([self.pool(feats[b, s]) for s in range(NS)])
            for b in range(B)
        ])                                                            # (B, NS, dim)
        pooled = pooled * mask.unsqueeze(-1)      # zero out absent slots
        if self.training and self.slot_dropout > 0:
            drop = (torch.rand_like(mask) > self.slot_dropout).float()
            # never drop a study's last remaining slot
            drop = torch.where((mask * drop).sum(1, keepdim=True) > 0,
                               drop, torch.ones_like(drop))
            mask = mask * drop
            pooled = pooled * mask.unsqueeze(-1)
        if self.head_type == "attn":
            return self.attn_head(self.drop(pooled), mask)
        x = torch.cat([pooled.reshape(B, -1), mask], dim=1)
        return self.head(self.drop(x))

    def forward_windows(self, arr, centres, slot_id, study_ix, pos, slot_starts):
        """P-25 window mode, B studies per call (P-32). arr (B, T, P, P) uint8 on the device (c02 flat) or
        (1, 6, S, P, P) (c01 dense, one study only); centres / slot_id / study_ix / pos are flat (W_total,)
        long tensors: each window's centre inside its slot's stack, its slot, the study it belongs to and
        its index within that study (collate_windows). Gathers [c-1, c, c+1] triplets, scales, resizes to
        img_size, augments (training, `aug`), ImageNet-normalises ON THE GPU, runs the encoder over EVERY
        window of the batch in one pass (the BatchNorm batch), then scatters the features into a
        (B, W_max, dim) tensor with a validity mask for the window head."""
        if arr.ndim == 5:                                   # c01 dense (B, 6, S, P, P)
            if arr.shape[0] != 1:
                raise SystemExit("c01 dense arrays support batch_studies=1 only (no c01 window member exists)")
            S = arr.shape[2]
            starts = torch.arange(arr.shape[1], device=arr.device) * S
            arr = arr.reshape(arr.shape[0], -1, *arr.shape[3:])   # (1, 6*S, P, P)
        else:
            starts = torch.as_tensor(slot_starts, device=arr.device, dtype=torch.long)
        B = arr.shape[0]
        base = starts[slot_id] + centres                    # (W,) row of each centre in its study's array
        idx = torch.stack([base - 1, base, base + 1], dim=1)  # (W, 3)
        x = arr[study_ix.unsqueeze(1), idx].float() / 255.0  # (W, 3, P, P)
        if x.shape[-1] != self.img_size:
            x = F.interpolate(x, size=(self.img_size, self.img_size), mode="bilinear",
                              align_corners=False)
        if self.training and self.aug != "none":            # P-33: draws nothing when aug == "none"
            x = augment_light(x)
        x = (x - IMAGENET_MEAN.to(x.device)) / IMAGENET_STD.to(x.device)
        if self.training and torch.rand(()) < 0.5:
            x = x + torch.randn_like(x) * 0.01              # the Dataset's noise aug, moved here
        feats = self.encode(x)                              # (W, dim) -- one pass over every study's windows
        if self.head_type != "window_attn":
            raise SystemExit("window_mode='random' needs head_type='window_attn'")
        n_per = torch.bincount(study_ix, minlength=B)
        w_max = max(int(n_per.max()) if n_per.numel() else 0, 1)
        padded = feats.new_zeros(B, w_max, feats.shape[-1])
        valid = torch.zeros(B, w_max, dtype=torch.bool, device=feats.device)
        sid_p = torch.zeros(B, w_max, dtype=torch.long, device=feats.device)   # 0, never -1: masked anyway
        padded[study_ix, pos] = feats
        valid[study_ix, pos] = True
        sid_p[study_ix, pos] = slot_id
        return self.window_head(self.drop(padded), sid_p, valid)


def weighted_bce(logits, y, w):
    """Confidence-weighted soft-target BCE, normalised PER STUDY then averaged over the batch.

    Per study on purpose (P-32): with batch_studies > 1 a single `Σ w·bce / Σ w` over the batch would let
    a gold study (weight 8) swallow its partner's gradient; normalising each row first keeps every
    study's contribution what it was at batch 1 (identical to the old formula for B = 1).
    No `pos_weight`: with soft targets it inflates every prediction and the metric
    reads only rank order, so there is nothing to gain and a collapse to overprediction
    to lose.
    """
    loss = F.binary_cross_entropy_with_logits(logits, y, reduction="none")
    per_study = (loss * w).sum(1) / w.sum(1).clamp_min(1e-6)
    return per_study.mean()


def build_model(c, device):
    """One factory for training and inference, from a Config or a checkpoint's saved config."""
    g = _cfg_get(c)
    backbone = g("backbone", "dinov2")
    sm = g("stack_mode", "triplet")
    in_ch = int(g("cache_n_slices", 16)) if sm == "channels" else 3
    m = KneeNet(resolve_backbone_dir(backbone), dropout=float(g("dropout", 0.1)),
                head_type=g("head_type", "concat"), slot_dropout=float(g("slot_dropout", 0.0)),
                backbone=backbone, in_chans=in_ch, slot_embed=bool(g("slot_embed", True)),
                grad_checkpoint=bool(g("grad_checkpoint", False)), img_size=int(g("img_size", 224)),
                aug=str(g("aug", "none")))          # old checkpoints predate the field -> "none"
    return m.to(device)


def collate_windows(items):
    """P-32 collate for window-mode studies (batch_studies >= 1). Stacks the fixed-shape uint8 arrays to
    (B, T, P, P), concatenates every study's (centre, slot) windows into flat tensors with `study_ix`
    (which study each window belongs to) and `pos` (its index within that study), stacks mask / y / w /
    is_gold and keeps the study list. One code path serves B = 1 (evaluation, inference) and B > 1."""
    out = {"study": [it["study"] for it in items],
           "arr": torch.stack([it["arr"] for it in items]),
           "centres": torch.cat([it["centres"] for it in items]),
           "slot_id": torch.cat([it["slot_id"] for it in items]),
           "study_ix": torch.cat([torch.full((len(it["centres"]),), i, dtype=torch.long)
                                  for i, it in enumerate(items)]),
           "pos": torch.cat([torch.arange(len(it["centres"]), dtype=torch.long) for it in items]),
           "mask": torch.stack([it["mask"] for it in items])}
    for k in ("y", "w", "is_gold"):
        if k in items[0]:
            out[k] = torch.stack([it[k] for it in items])
    return out


def forward_batch(model, b, device, cfg):
    """Logits for one batch, whichever representation the Dataset produced: fixed windows
    (`imgs`, one view) or random/all windows (`arr` + indices through collate_windows). TTA views are
    NOT handled here (training only); predict_probs() does the multi-view pooling."""
    if "arr" in b:
        if "study_ix" not in b or "pos" not in b or b["centres"].ndim != 1:
            raise SystemExit("window batches must come through collate_windows (flat centres + study_ix / pos); "
                             "a default-collated window batch would be misread -- attach collate_fn=collate_windows")
        _, _, slot_slices, _ = cache_geom(cfg)
        starts, _ = slot_offsets(slot_slices)
        return model.forward_windows(b["arr"].to(device), b["centres"].to(device), b["slot_id"].to(device),
                                     b["study_ix"].to(device), b["pos"].to(device), starts)
    imgs = b["imgs"]
    if imgs.ndim == 7:                                   # (B, n_views, 6, K, 3, H, W): view 0 only
        imgs = imgs[:, 0]
    return model(imgs.to(device), b["mask"].to(device))


FOCAL_MAX = {"Fracture", "Contusion", "Medial Meniscus", "Lateral Meniscus", "Baker's"}
FOCAL_TOP2 = {"ACL", "MCL"}


def pool_views(probs, how):
    """(n_views, B, L) probabilities -> (B, L). "mean" averages; "focal" is the 0.936 notebook's
    per-label rule (max for focal findings, top-2 mean for the cruciate/collateral, mean else)."""
    if probs.shape[0] == 1 or how == "mean":
        return probs.mean(0)
    out = probs.mean(0).clone()
    for i, lab in enumerate(LABELS):
        if lab in FOCAL_MAX:
            out[:, i] = probs[:, :, i].max(0).values
        elif lab in FOCAL_TOP2:
            k = min(2, probs.shape[0])
            out[:, i] = probs[:, :, i].topk(k, dim=0).values.mean(0)
    return out


@torch.no_grad()
def predict_probs(model, b, device, cfg):
    """Per-study probabilities with the member's TTA applied: for fixed-window members the
    Dataset stacks one view per `tta_offsets` entry along a leading axis; each view is a forward
    pass and the views are pooled per label with `tta_pool`. (0,) + "mean" == a single forward."""
    if "arr" in b or b["imgs"].ndim != 7:
        return torch.sigmoid(forward_batch(model, b, device, cfg)).float()
    views = []
    for v in range(b["imgs"].shape[1]):
        logits = model(b["imgs"][:, v].to(device), b["mask"].to(device))
        views.append(torch.sigmoid(logits).float())
    return pool_views(torch.stack(views), getattr(cfg, "tta_pool", "mean"))

## Section 7: training

Built around one operational fact: **five folds do not fit in one 9-hour Kaggle
session.** So every fold writes a resumable `*_last.pt` after each epoch, the
runtime guard stops cleanly before the ceiling, and re-running with the previous
output attached picks up where it left off. A run that cannot resume wastes a
whole session.

Also here: AMP, gradient accumulation (batch of 1 study is already ~36 ViT
forwards), cosine schedule with warmup, gradient clipping, and a
**prediction-spread diagnostic**. That last one exists because the known failure
mode of this setup is collapse to the base rate — every study gets the same score,
AUC 0.5, and the loss looks fine. Near-zero spread is an alarm, never a target.

In [ ]:
# ── Section 7: training ───────────────────────────────────────────────────────
def seed_worker(worker_id):
    """Re-seed numpy and `random` inside each DataLoader worker.

    PyTorch seeds only torch's RNG per worker; numpy and `random` are inherited from the
    parent by fork. Workers are recreated every epoch from the same parent state, so
    without this the "random" slice jitter (P-08) and the Gaussian noise are byte-identical
    in every epoch -- augmentation that never augments. `torch.initial_seed()` inside a
    worker is base_seed + worker_id, and base_seed advances each epoch.
    """
    s = torch.initial_seed() % (2 ** 32)
    np.random.seed(s)
    random.seed(s)


def check_worker_rng():
    """Direct test of traps 6e on THIS platform, in seconds.

    Linux forks DataLoader workers from a parent whose numpy/`random` state has not moved
    between epochs, so without a `worker_init_fn` every epoch draws the same "random"
    numbers and slice jitter never jitters. Windows spawns instead, so this cannot be
    reproduced locally -- which is exactly why the check runs on Kaggle and prints both
    arms. Expect: without = True (identical, the bug), with = False (varying, fixed).
    """
    class _Probe(Dataset):
        def __len__(self):
            return 4

        def __getitem__(self, i):
            return torch.tensor([np.random.randint(0, 10 ** 6), random.randint(0, 10 ** 6)])

    print("  worker RNG check (traps 6e):")
    for label, init in (("without worker_init_fn", None), ("with seed_worker", seed_worker)):
        try:
            dl = DataLoader(_Probe(), batch_size=4, num_workers=2, worker_init_fn=init)
            eps = [torch.cat([b for b in dl]).flatten().tolist() for _ in range(3)]
            same = eps[0] == eps[1] == eps[2]
            print(f"    {label:<24} identical across 3 epochs = {same}"
                  f"   {'<-- augmentation would never vary' if same else ''}")
        except Exception as e:
            print(f"    {label:<24} check failed: {type(e).__name__}: {e}")


def split_studies(targets, fold, cfg):
    """(train, val) StudyInstanceUIDs for one fold. train_all (P-28): every non-gold row of every
    fold trains, the gold rows are the validation set -- there is no OOF for such a member."""
    if getattr(cfg, "train_all", False):
        tr = targets.loc[targets.is_gold == 0, "StudyInstanceUID"].tolist()
        va = targets.loc[targets.is_gold == 1, "StudyInstanceUID"].tolist()
    else:
        tr = targets.loc[targets.fold != fold, "StudyInstanceUID"].tolist()
        va = targets.loc[targets.fold == fold, "StudyInstanceUID"].tolist()
    return tr, va


def make_loaders(manifest, targets, image_root, cfg, fold):
    tr_studies, va_studies = split_studies(targets, fold, cfg)
    if cfg.smoke:
        avail = set(manifest.StudyInstanceUID)
        tr_studies = [s for s in tr_studies if s in avail][:4]
        # train_all: a few gold rows, so the AUC has both classes on some labels
        va_studies = [s for s in va_studies if s in avail][:(8 if cfg.train_all else 4)]
        if not tr_studies:      # local sample has no training studies at all
            tr_studies = va_studies = sorted(avail)[:3]
        if not va_studies:
            # train_all locally: the 3 placeholder rows are non-gold, so there is no gold row to
            # hold out. Without this, evaluate() returns ({}, None), the score silently falls back
            # to -loss, no _oof.csv is written and the SWA evaluation is never exercised.
            print("  smoke/train_all: no gold study in the local sample -> val = train")
            va_studies = tr_studies
    tr_ds = KneeStudyDataset(manifest, targets, image_root, cfg, True, tr_studies)
    va_ds = KneeStudyDataset(manifest, targets, image_root, cfg, False, va_studies)
    print(f"  fold {fold}: train {len(tr_ds)} / val {len(va_ds)} studies"
          + (" [train_all: val = gold rows]" if cfg.train_all else ""))
    nw = 0 if cfg.smoke else cfg.num_workers
    # Window-mode items travel through collate_windows (P-32) at any batch size; evaluation is always ONE
    # study per batch, so the OOF path is bit-identical whatever batch_studies the arm trains with.
    collate = collate_windows if getattr(cfg, "window_mode", "fixed") == "random" else None
    return (DataLoader(tr_ds, batch_size=cfg.batch_studies, shuffle=True,
                       num_workers=nw, drop_last=False, worker_init_fn=seed_worker, collate_fn=collate),
            DataLoader(va_ds, batch_size=1, shuffle=False,
                       num_workers=nw, collate_fn=collate))


def bootstrap_macro_ci(Y_hard, P, n_boot=2000, seed=0):
    """Percentile-bootstrap 95% CI of the macro-AUC over studies. With ~12 gold
    studies per fold this interval is enormous -- which is the point of printing it."""
    rng = np.random.default_rng(seed)
    n = len(P)
    if n < 4:
        return (float("nan"), float("nan"))
    vals = []
    for _ in range(n_boot):
        ix = rng.integers(0, n, n)
        a = [auc_score(Y_hard[ix, i], P[ix, i]) for i in range(len(LABELS))]
        a = [v for v in a if np.isfinite(v)]
        if a:
            vals.append(float(np.mean(a)))
    if not vals:
        return (float("nan"), float("nan"))
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)))


def evaluate(model, loader, device, cfg):
    """Validation pass. Returns (metrics, table) where `table` is a DataFrame with the
    per-study predictions, targets, weights and gold flag -- the OOF rows. Per-label
    numbers are kept because the metric charges every label the same, so the label
    stuck at 0.5 is the thing we most need to see. TTA (tta_offsets / tta_pool, eval_windows)
    is whatever `cfg` says -- oof_eval and infer must run the same setting."""
    model.eval()
    P, Y, W, G, S = [], [], [], [], []
    with torch.no_grad():
        for b in loader:
            P.append(predict_probs(model, b, device, cfg).cpu().numpy())
            Y.append(b["y"].numpy())
            W.append(b["w"].numpy())
            G.append(b["is_gold"].numpy())
            S.extend(b["study"])
    if not P:
        return {}, None
    P, Y, W, G = (np.concatenate(x) for x in (P, Y, W, G))
    hard = (Y > 0.5).astype(int)
    gm = G > 0.5

    per_label = {}
    for i, lab in enumerate(LABELS):
        row = {"auc_soft": auc_score(hard[:, i], P[:, i]),
               "pred_std": float(P[:, i].std())}
        if gm.sum() >= 4:
            row["auc_gold"] = auc_score(hard[gm, i], P[gm, i])
        per_label[lab] = row

    def macro(key):
        vals = [r[key] for r in per_label.values() if np.isfinite(r.get(key, np.nan))]
        return round(float(np.mean(vals)), 4) if vals else float("nan")

    out = {"pred_std": round(float(P.std(0).mean()), 4),
           "auc_soft": macro("auc_soft"),
           "n_labels_scored": int(sum(np.isfinite(r["auc_soft"]) for r in per_label.values()))}
    if gm.sum() >= 4:
        out["auc_gold"] = macro("auc_gold")
        out["n_gold"] = int(gm.sum())
        lo, hi = bootstrap_macro_ci(hard[gm], P[gm])
        out["auc_gold_ci95"] = (round(lo, 3), round(hi, 3))
    out["per_label"] = per_label

    table = pd.DataFrame({"StudyInstanceUID": S, "is_gold": G.astype(int)})
    for i, lab in enumerate(LABELS):
        table[f"pred__{lab}"] = P[:, i]
        table[f"y__{lab}"] = Y[:, i]
        table[f"w__{lab}"] = W[:, i]
    return out, table


def print_per_label(per_label):
    print(f"    {'label':<18} {'auc_soft':>8} {'auc_gold':>8} {'pred_std':>8}")
    for lab, r in per_label.items():
        g = r.get("auc_gold", float("nan"))
        print(f"    {lab:<18} {r['auc_soft']:8.3f} {g:8.3f} {r['pred_std']:8.3f}"
              + ("   <-- near chance" if np.isfinite(r["auc_soft"]) and r["auc_soft"] < 0.55 else "")
              + ("   <-- collapsed" if r["pred_std"] < 0.01 else ""))


def param_groups(model, cfg):
    """Layer-wise LR decay for the DINOv2 encoder + no weight decay on 1-D params.

    HF Dinov2Model parameter names look like `embeddings.*`, `encoder.layer.<i>.*`,
    `layernorm.*`. The top block and the final LayerNorm get `lr_backbone`; each block
    below gets one more factor of `llrd_decay`; embeddings one more still. The head
    and the attention pool are freshly initialised, so they get `lr_head` undecayed.
    """
    # DINOv2: `encoder.layer.<i>` x 12 blocks. ConvNeXt (HF): `encoder.stages.<s>` x 4 stages
    # (depths 3/3/9/3) -- decay per stage, since a stage is the CNN's unit of feature level.
    # timm hybrids (coatnet_rmlp_*): `stem.*`, `stages.<s>.*` x 4, `norm.*` -- same per-stage rule.
    is_cnn = getattr(model, "backbone", "dinov2") == "convnext_tiny"
    is_timm = str(getattr(model, "backbone", "dinov2")).startswith("timm:")
    if is_timm:
        n_blocks = len(model.enc.stages)
    else:
        n_blocks = (len(model.enc.config.hidden_sizes) if is_cnn
                    else model.enc.config.num_hidden_layers)
    groups = {}

    def add(name, p, lr):
        no_decay = (p.ndim == 1 or name.endswith(".bias") or "token" in name
                    or "position_embeddings" in name)       # BEiT/MAE convention
        key = (round(lr, 12), no_decay)
        groups.setdefault(key, {"params": [], "lr": lr,
                                "weight_decay": 0.0 if no_decay else cfg.weight_decay})
        groups[key]["params"].append(p)

    for name, p in model.enc.named_parameters():
        if not p.requires_grad:
            continue
        if getattr(model, "in_chans", 3) != 3 and "patch_embeddings" in name:
            add(name, p, cfg.lr_stem)     # widened conv = new capacity; under LLRD it would never move
            continue
        if name.startswith("embeddings.") or name.startswith("stem."):
            depth = 0
        elif name.startswith("encoder.layer.") or name.startswith("encoder.stages."):
            depth = int(name.split(".")[2]) + 1
        elif name.startswith("stages."):                 # timm: stages.<s>.blocks.<j>...
            depth = int(name.split(".")[1]) + 1
        else:                       # final layernorm
            depth = n_blocks + 1
        lr = cfg.lr_backbone * (cfg.llrd_decay ** (n_blocks + 1 - depth))
        add(name, p, lr)
    # Everything that is not the encoder is freshly initialised and gets lr_head undecayed.
    # Enumerated by name rather than hard-coded, so P-09's `attn_head` cannot silently end
    # up with no optimizer group when head_type="attn".
    n_head = 0
    for mname, mod in model.named_children():
        if mname == "enc":
            continue
        for name, p in mod.named_parameters():
            add(f"{mname}.{name}", p, cfg.lr_head)
            n_head += p.numel()
    out = list(groups.values())
    lrs = sorted({g["lr"] for g in out if g["lr"] < cfg.lr_head})
    print(f"  backbone LR range {lrs[0]:.2e} .. {lrs[-1]:.2e} over {n_blocks} blocks "
          f"(decay {cfg.llrd_decay}); head {cfg.lr_head:.0e} over {n_head:,} params "
          f"(head_type={getattr(model, 'head_type', 'concat')})")
    return out


class EMA:
    """Exponential moving average of the weights. Validated and saved instead of the
    raw weights: it is markedly more robust to label noise and makes a fixed epoch
    count a safe selection rule. Buffers are copied, not averaged."""

    def __init__(self, model, decay):
        import copy
        self.decay = decay
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, e in self.module.state_dict().items():
            m = msd[k]
            if e.dtype.is_floating_point:
                e.mul_(self.decay).add_(m.detach(), alpha=1 - self.decay)
            else:
                e.copy_(m)


def average_state_dicts(sds):
    """Element-wise mean of N state_dicts (SWA, P-28): float tensors averaged in fp32 and cast
    back to their dtype; everything else (BatchNorm num_batches_tracked, int buffers) copied from
    the LAST one. Averaging BatchNorm running stats is an approximation; three adjacent EMA
    snapshots are close enough that it holds, and the `_lastema.pt` vs `_best.pt` print is the check."""
    out = {}
    for k, v in sds[-1].items():
        if v.dtype.is_floating_point:
            out[k] = torch.stack([sd[k].float() for sd in sds]).mean(0).to(v.dtype)
        else:
            out[k] = v.clone()
    return out


def train_fold(fold, manifest, targets, image_root, cfg, device):
    ckpt_best = os.path.join(WORK, f"{cfg.version}_fold{fold}_best.pt")
    ckpt_last = os.path.join(WORK, f"{cfg.version}_fold{fold}_last.pt")
    ckpt_lastema = os.path.join(WORK, f"{cfg.version}_fold{fold}_lastema.pt")
    oof_path = os.path.join(WORK, f"{cfg.version}_fold{fold}_oof.csv")

    model = build_model(cfg, device)
    opt = torch.optim.AdamW(param_groups(model, cfg))
    ema = EMA(model, cfg.ema_decay) if cfg.ema_decay > 0 else None

    tr_loader, va_loader = make_loaders(manifest, targets, image_root, cfg, fold)
    steps_per_epoch = max(1, len(tr_loader) // cfg.grad_accum)
    total = steps_per_epoch * cfg.epochs
    warm = max(1, int(total * cfg.warmup_frac))

    def lr_at(step):
        if step < warm:
            return step / warm
        p = (step - warm) / max(1, total - warm)
        return 0.5 * (1 + math.cos(math.pi * min(p, 1.0)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)
    use_amp = cfg.amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    start_epoch, best, best_epoch = 0, -1.0, -1
    swa_ring = []           # EMA snapshots of the last `swa_last` completed epochs (CPU)
    # A smoke run never resumes: a stale `_last.pt` from an earlier local smoke made a
    # 1-epoch smoke "resume at epoch 1 of 1", skip training entirely and still finish
    # green -- the checkpoint code it was meant to exercise never ran (traps 19).
    if os.path.exists(ckpt_last) and not cfg.smoke:
        st = torch.load(ckpt_last, map_location=device, weights_only=False)
        model.load_state_dict(st["model"])
        if ema is not None:
            # a checkpoint without an EMA (or with EMA switched on later) must not
            # leave the EMA copy at its random-head initialisation
            ema.module.load_state_dict(st.get("ema", st["model"]))
        opt.load_state_dict(st["opt"])
        sched.load_state_dict(st["sched"])
        start_epoch = st["epoch"] + 1
        best = st.get("best", -1.0)
        best_epoch = st.get("best_epoch", st["epoch"])
        print(f"  resumed fold {fold} at epoch {start_epoch} (best {best:.4f} at epoch {best_epoch})")
        if cfg.swa_last > 0:
            swa_ring = [{k: v.detach().to("cpu") for k, v in sd.items()} for sd in st.get("swa_ring", [])]
            if len(swa_ring) < min(cfg.swa_last, start_epoch):
                print(f"  ! resumed with {len(swa_ring)} SWA snapshot(s) in _last.pt; the average "
                      f"will cover fewer than swa_last={cfg.swa_last} epochs")
        del st

    for epoch in range(start_epoch, cfg.epochs):
        model.train()
        running, nb = 0.0, 0
        t_epoch = time.time()
        n_studies = 0
        guard_hit = False
        opt.zero_grad(set_to_none=True)
        for i, b in enumerate(tr_loader):
            with torch.amp.autocast("cuda", enabled=use_amp):
                logits = forward_batch(model, b, device, cfg)
                loss = weighted_bce(logits, b["y"].to(device), b["w"].to(device))
            scaler.scale(loss / cfg.grad_accum).backward()
            if (i + 1) % cfg.grad_accum == 0:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                sched.step()
                if ema is not None:
                    ema.update(model)
                if epoch == start_epoch and (i + 1) == cfg.grad_accum and device.type == "cuda":
                    # P-32: batch_studies x train_windows memory is unmeasured on a 15 GB T4; say it early
                    print(f"    peak GPU memory after the first optimiser step: "
                          f"{torch.cuda.max_memory_allocated() / 2**30:.2f} GiB "
                          f"(batch {cfg.batch_studies} x {cfg.train_windows} windows, accum {cfg.grad_accum})")
            running += float(loss.detach())
            nb += 1
            n_studies += int(b["mask"].shape[0])
            # Throughput is the open risk of this pipeline; print it early and often.
            if n_studies in (10, 50) or (n_studies % 500 == 0):
                dt = time.time() - t_epoch
                geom_note = (f"windows/study {cfg.train_windows}" if cfg.window_mode == "random"
                             else f"slices/slot {cfg.slices_per_slot}")
                print(f"    {n_studies} studies in {dt:.0f}s = {dt/n_studies:.2f} s/study "
                      f"({geom_note}, img {cfg.img_size}, workers "
                      f"{tr_loader.num_workers}) -> epoch ETA "
                      f"{dt/n_studies*len(tr_loader.dataset)/60:.0f} min")
            if out_of_time():
                print("  runtime guard hit mid-epoch")
                guard_hit = True
                break
        train_secs = time.time() - t_epoch

        eval_model = ema.module if ema is not None else model
        if cfg.swa_last > 0 and ema is not None and not guard_hit:
            # a partial epoch (guard fired mid-way) is not a converged point on the trajectory
            swa_ring = (swa_ring + [{k: v.detach().to("cpu", copy=True)
                                     for k, v in ema.module.state_dict().items()}])[-cfg.swa_last:]
        t_eval = time.time()
        metrics, oof = evaluate(eval_model, va_loader, device, cfg)
        per_label = metrics.pop("per_label", {})
        print(f"  fold {fold} epoch {epoch}: loss {running/max(nb,1):.4f}  {metrics}")
        print(f"    train {train_secs/60:.1f} min ({train_secs/max(n_studies,1):.2f} s/study), "
              f"val {(time.time()-t_eval)/60:.1f} min")
        if per_label:
            print_per_label(per_label)
        if metrics.get("pred_std", 1.0) < 0.01:
            print("  !! prediction spread near zero -- base-rate collapse, not a "
                  "converged model")

        # Which epoch is "the" model? Selecting on the ~11 gold studies per fold is a coin
        # flip (Hanley-McNeil SE ~0.09) and stays banned. Through v05 `_best.pt` was simply
        # the EMA weights after the LAST completed epoch (fixed-epoch, P-03/P-04). P-22
        # (src/oof_epoch_analysis.py, 2026-08-29) then measured selection on OOF-vs-teacher
        # over the 882 held-out studies: +0.013 split-half for the concat head, which peaks
        # mid-schedule and decays, ~0 for the attention head, gold flat at the chosen epoch --
        # so `ckpt_policy="best_oof"` keeps the epoch with the highest auc_soft so far.
        # The score is never gold. A NaN score cannot drop a fold: the first epoch is always
        # written, and an undefined AUC falls back to the loss.
        score = metrics.get("auc_soft")
        if score is None or not np.isfinite(score):
            score = -running / max(nb, 1)
        take = (cfg.ckpt_policy == "last" or score > best
                or not os.path.exists(ckpt_best))
        if take:
            best, best_epoch = score, epoch
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(),
                    "sched": sched.state_dict(), "epoch": epoch, "best": best,
                    "best_epoch": best_epoch,
                    **({"ema": ema.module.state_dict()} if ema is not None else {}),
                    **({"swa_ring": swa_ring} if cfg.swa_last > 0 else {})},
                   ckpt_last)
        if oof is not None:
            oof.insert(1, "epoch", epoch)
            oof.to_csv(oof_path.replace("_oof.csv", f"_ep{epoch}_oof.csv"), index=False)
        if take:
            torch.save({"model": eval_model.state_dict(), "score": score, "epoch": epoch,
                        "ema": ema is not None, "config": asdict(cfg)}, ckpt_best)
            if oof is not None:
                oof.to_csv(oof_path, index=False)        # always the checkpointed epoch
        print(f"    epoch {epoch} EMA score {score:.4f} -> "
              + (f"checkpoint = epoch {epoch} ({os.path.basename(ckpt_best)} + "
                 f"{os.path.basename(oof_path)})" if take else
                 f"not taken; best.pt stays epoch {best_epoch} ({best:.4f})")
              + f" [ckpt_policy={cfg.ckpt_policy}]")

        if out_of_time():
            print("  stopping: runtime guard. Attach this output and re-run to resume.")
            return model, best, False

    if cfg.swa_last > 0 and ema is not None and swa_ring:
        # P-28: `_best.pt` becomes the average of the last N EMA snapshots; the final-epoch EMA
        # (what policy "last" just wrote) is kept beside it for the A/B. Same keys as every other
        # `_best.pt`, so member_settings() and the infer loader need no change.
        shutil.copyfile(ckpt_best, ckpt_lastema)
        swa_sd = average_state_dicts(swa_ring)
        ema.module.load_state_dict(swa_sd)
        t_eval = time.time()
        metrics, oof = evaluate(ema.module, va_loader, device, cfg)
        per_label = metrics.pop("per_label", {})
        score = metrics.get("auc_soft", float("nan"))
        print(f"  fold {fold} SWA of last {len(swa_ring)} EMA snapshot(s): {metrics}  "
              f"(last-epoch EMA scored {best:.4f}; val {(time.time()-t_eval)/60:.1f} min)")
        if per_label:
            print_per_label(per_label)
        torch.save({"model": swa_sd, "score": score, "epoch": cfg.epochs - 1, "ema": True,
                    "swa_last": len(swa_ring), "config": asdict(cfg)}, ckpt_best)
        if oof is not None:
            oof.insert(1, "epoch", cfg.epochs - 1)
            oof.to_csv(oof_path, index=False)
        print(f"    -> {os.path.basename(ckpt_best)} = SWA, {os.path.basename(ckpt_lastema)} = last EMA")
        del swa_ring

    return model, best, True

## Section 8: run

On Kaggle this trains the configured folds; locally (`smoke=True`) it runs one
fold over the 3 sample studies purely to prove the loop executes.

In [ ]:
# ── Section 8: run training ───────────────────────────────────────────────────
if os.environ.get("RSNA_DEFS_ONLY"):
    raise SystemExit(0)          # src/cache_selftest.py imports Sections 1-7 and stops here

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

def resolve_image_root(series_csv: str, default_root: str) -> str:
    """Find the directory that actually holds `<study>/<series>/` for this CSV.

    Submission #1 (kernel v2, smoke) scored exactly 0.500 on the hidden test, which
    is what a constant submission scores -- i.e. on the rerun no test study was
    found under the assumed root and the 0.5 fallback fired, silently. Probing the
    tree beats assuming it, and failing loudly beats a silent 0.5 (see below).
    """
    meta = pd.read_csv(series_csv)
    if len(meta) == 0:
        return default_root
    first = meta.iloc[0]
    if os.path.isdir(os.path.join(default_root, first.StudyInstanceUID,
                                  first.SeriesInstanceUID)):
        return default_root
    # Shallow probe: <COMP>/<x>/<study>/<series> and one level deeper. Never `**` --
    # that walks the whole ~819k-file mount.
    hits = shallow_glob(COMP, first.SeriesInstanceUID, max_depth=3, skip=("train_series",))
    if not hits and ON_KAGGLE:
        hits = shallow_glob("/kaggle/input", first.SeriesInstanceUID, max_depth=4,
                            skip=("train_series",))
    if hits:
        root = os.path.dirname(os.path.dirname(hits[0]))
        print(f"  ! image root for {os.path.basename(series_csv)} is not {default_root}"
              f" -- found {root}")
        return root
    print(f"  ! could not locate any series of {os.path.basename(series_csv)} "
          f"under {default_root} or by glob")
    return default_root


TRAIN_IMG = os.path.join(COMP, "train_series")
TEST_IMG = os.path.join(COMP, "test_series")
if not os.path.isdir(TRAIN_IMG) and os.path.isdir(os.path.join(COMP, "sample_dicom",
                                                               "test_series")):
    # Local: only the public test tree exists, so use it for both.
    TRAIN_IMG = TEST_IMG = os.path.join(COMP, "sample_dicom", "test_series")
else:
    TEST_IMG = resolve_image_root(os.path.join(COMP, "test_series.csv"), TEST_IMG)
print(f"train images: {TRAIN_IMG}\ntest images:  {TEST_IMG}")

# ---- which mode are we in? ----------------------------------------------------
def find_mounted_checkpoints(version, kind="best"):
    """`{version}_fold<k>_{kind}.pt` files attached as a kernel/dataset input (Kaggle) or
    left in artifacts/kaggle_out (local). Shallow search only. Returns {fold: path}."""
    import re
    # Locally, WORK (this machine's own smoke checkpoints) is searched only when MODE asks for
    # inference explicitly -- in "auto" it would flip every local smoke run into infer mode.
    roots = (["/kaggle/input"] if ON_KAGGLE else
             ["artifacts/kaggle_out"] + ([WORK] if MODE in ("infer", "oof_eval") else []))
    found = {}
    for root in roots:
        # depth 4 like load_cache_manifests: a new slug mounts kernel outputs type-prefixed
        # (/kaggle/input/<type>/<owner>/<name>/...), an old one at /kaggle/input/<name>/ (traps 6f)
        for p in shallow_glob(root, f"{version}_fold*_{kind}.pt", max_depth=4):
            m = re.search(rf"{re.escape(version)}_fold(\d+)_{kind}\.pt$", p)
            if m:
                found.setdefault(int(m.group(1)), p)
    return found


mounted_ckpts = find_mounted_checkpoints(cfg.version, "best")
mounted_last = find_mounted_checkpoints(cfg.version, "last")
if MODE != "auto":
    mode = MODE
else:
    # infer only when EVERY configured fold has a finished checkpoint; a partial run
    # (guard fired) must resume training, not be submitted.
    mode = "infer" if mounted_ckpts and set(cfg.folds) <= set(mounted_ckpts) else "train"
print(f"MODE={mode}  mounted best: {sorted(mounted_ckpts)}  mounted last: {sorted(mounted_last)}")

# What a member's checkpoint decides, split in two (2026-08-30). CACHE keys describe the decoded
# test array -- members that agree on all of them share ONE decode-once pass (a "geometry group");
# c01 members (v05a/v05b/v05g/v06c) and c02 members (v08w, the hybrids) are two groups in one
# blend. MEMBER keys only change how a member READS the array and are applied per member around
# predict() -- the way stack_mode already was (P-21 heads, P-23 stack, P-25 windows, P-12 TTA).
INFER_CACHE_KEYS = ("use_cache", "cache_scheme", "cache_px", "cache_n_slices", "cache_px_wide",
                    "cache_slot_slices", "cache_band", "crop_mm", "lat_dead_zone_mm")
INFER_MEMBER_KEYS = ("slices_per_slot", "triplet_gap", "img_size", "stack_mode", "lat_undo",
                     "window_mode", "eval_windows", "tta_offsets", "tta_pool", "head_type",
                     "backbone", "slot_embed", "dropout", "slot_dropout")


def _norm_val(v):
    return tuple(v) if isinstance(v, (list, tuple)) else v


def member_settings(saved, version=None):
    """Every CACHE + MEMBER key for one checkpoint: the saved config where present, else the
    dataclass default (old checkpoints predate the new fields and mean the c01-era value).
    INFER_OVERRIDES[version] then applies on top -- MEMBER keys only, TTA/eval_windows for
    members whose checkpoints predate them; it can never change what array is decoded."""
    out = {}
    for k in INFER_CACHE_KEYS + INFER_MEMBER_KEYS:
        if k in saved:
            out[k] = _norm_val(saved[k])
        else:
            out[k] = _norm_val(Config.__dataclass_fields__[k].default)
    for k, v in (INFER_OVERRIDES.get(version, {}) if version else {}).items():
        if k not in INFER_MEMBER_KEYS:
            raise SystemExit(f"INFER_OVERRIDES[{version}][{k}]: only member keys may be "
                             f"overridden at inference ({INFER_MEMBER_KEYS})")
        out[k] = _norm_val(v)
    return out


def cache_signature(settings):
    return tuple((k, settings[k]) for k in INFER_CACHE_KEYS)


def apply_settings(target_cfg, settings, keys):
    """setattr the chosen keys onto a Config (the module global, at inference); returns the
    previous values so they can be restored."""
    prev = {k: getattr(target_cfg, k) for k in keys}
    for k in keys:
        setattr(target_cfg, k, settings[k])
    return prev


infer_members = []          # [(version, fold, path)] -- the blend, in infer / oof_eval mode
infer_settings = {}         # (version, fold) -> resolved CACHE + MEMBER settings
infer_saved_cfg = {}        # (version, fold) -> the raw config dict saved in the checkpoint
if mode in ("infer", "oof_eval"):
    # P-21: the submission is a rank-mean over every mounted fold checkpoint of every version in
    # INFER_MEMBERS. Each version must be present -- a blend that silently lost a member is not
    # the model that was validated (the traps 6d failure class again). oof_eval scores fold 0
    # of each version on its held-out studies instead of predicting the test set.
    for v in (list(INFER_MEMBERS) or [cfg.version]):
        found = find_mounted_checkpoints(v, "best")
        if mode == "oof_eval":
            found = {f: p for f, p in found.items() if f in ARM_FOLDS}
        if not found:
            raise SystemExit(f"MODE={mode} but no {v}_fold*_best.pt is mounted (INFER_MEMBERS="
                             f"{INFER_MEMBERS}). Attach the training run's output as a kernel "
                             f"input (kernel_sources), or drop {v} from INFER_MEMBERS on purpose.")
        infer_members += [(v, f, found[f]) for f in sorted(found)]
    print(f"  {mode} members ({len(infer_members)}): "
          + ", ".join(f"{v}/fold{f}" for v, f, _ in infer_members))
    # The checkpoints decide the input geometry, not FORCE_SMOKE: a smoke-mode infer would
    # otherwise feed 2 slices/slot to a model trained on 6 and pass every assert.
    for v, f, p in infer_members:
        st0 = torch.load(p, map_location="cpu", weights_only=False)
        s = member_settings(st0.get("config", {}), v)
        infer_settings[(v, f)] = s
        infer_saved_cfg[(v, f)] = dict(st0.get("config", {}))
        # Fail here, in seconds, if a member's backbone weights are not mounted -- not after
        # seven other members have already predicted (infer v9, 2026-08-30: the ConvNeXt
        # dataset was missing from the infer kernel's sources).
        resolve_backbone_dir(s["backbone"])
        del st0
    groups = {}
    for (v, f), s in infer_settings.items():
        groups.setdefault(cache_signature(s), []).append(f"{v}/fold{f}")
    print(f"  {len(groups)} geometry group(s) (one decode-once pass each):")
    for sig, members in groups.items():
        d = dict(sig)
        print(f"    {cache_version_for(d)} x{len(members)}: {', '.join(members)}")
    for (v, f), s in infer_settings.items():
        print(f"    {v}/fold{f}: {s['backbone']}, {s['head_type']}, {s['window_mode']}"
              + (f", eval_windows {s['eval_windows']}" if s['window_mode'] == 'random' else
                 f", K {s['slices_per_slot']}, tta {s['tta_offsets']}/{s['tta_pool']}")
              + f", img {s['img_size']}")
    cfg.folds = tuple(sorted({f for _, f, _ in infer_members}))
else:
    # Resume: a previous session's output is mounted read-only; copy its checkpoints
    # into WORK so train_fold finds them (otherwise every fold restarts at epoch 0).
    # This block serves ARMS = None runs only -- it looks up the DEFAULT config's version. Arms
    # get their own copy inside the arm loop (traps 31: until 2026-09-21 an arm's mounted
    # `_last.pt` was never copied and every resumed arm silently restarted at epoch 0).
    for fold in cfg.folds:
        for kind, src_map in (("last", mounted_last), ("best", mounted_ckpts)):
            src = src_map.get(fold)
            dst = os.path.join(WORK, f"{cfg.version}_fold{fold}_{kind}.pt")
            if src and not os.path.exists(dst):
                shutil.copy(src, dst)
                print(f"  resume: copied {os.path.basename(src)} into WORK")

# ---- the caches (P-01 c01 / 2026-08-30 c02): shards written by src/cache_pipeline.py -----
def load_cache_manifests():
    """{cache_version: manifest DataFrame with a `locator` column}. EVERY mounted shard of every
    scheme is indexed; which cache an arm or a member reads is decided by cache_version_for(its
    config), so a c01 and a c02 cache can be mounted side by side."""
    roots = ["/kaggle/input"] if ON_KAGGLE else ["artifacts/cache_local"]
    frames = {}
    for root in roots:
        # depth 4, not 2: a NEWLY created kernel mounts kernel outputs type-prefixed
        # (/kaggle/input/<type>/<owner>/<name>/...) while older kernels mount them at
        # /kaggle/input/<name>/. max_depth=2 found the cache in rsna-knee-train and
        # silently missed it in rsna-knee-folds -- nine hours of the wrong recipe.
        for mpath in shallow_glob(root, "manifest_shard*.csv", max_depth=4):
            m = pd.read_csv(mpath, dtype={"mask": str})
            if "cache_version" not in m.columns or len(m) == 0:
                print(f"  ! {mpath}: no cache_version column or empty, ignored")
                continue
            version = str(m.cache_version.iloc[0])
            m = m[m.get("cached", 1) == 1].copy()
            arr_dir = os.path.join(os.path.dirname(mpath), version)
            if "blob" in m.columns:                     # c02: (blob path, row inside the blob)
                m["locator"] = [(os.path.join(arr_dir, str(b)), int(r)) for b, r in zip(m.blob, m.row)]
                m = m[[os.path.exists(loc[0]) for loc in m.locator]]
            else:                                       # c01: one .npy per study
                m["locator"] = [os.path.join(arr_dir, f"{u}.npy") for u in m.StudyInstanceUID]
                m = m[[os.path.exists(x) for x in m.locator]]
            m["mask"] = m["mask"].map(lambda v: str(v).zfill(len(SLOTS)) if isinstance(v, str) or v == v else "")
            frames.setdefault(version, []).append(m)
            print(f"  cache shard {mpath}: {len(m)} studies ({version})")
    return {v: pd.concat(fs, ignore_index=True) for v, fs in frames.items()}


cache_manifests = load_cache_manifests() if cfg.use_cache else {}
for _v, _m in cache_manifests.items():
    CACHE_INDEX[_v] = dict(zip(_m.StudyInstanceUID, _m.locator))
    print(f"  cache: {len(CACHE_INDEX[_v])} studies indexed ({_v})")
if cfg.use_cache and not cache_manifests and mode == "infer":
    # `use_cache` selects the PREPROCESSING (130 mm crop, per-series 1/99 normalisation,
    # laterality) as well as the array read. No TEST study is ever in the cache, so infer
    # builds every study through build_study_array -- the same functions the cache was
    # built with. Flipping it off here would take the v02 decode branch and score a v03
    # model on v02 pixels, and nothing would say so (traps.md 12d).
    print("  infer: no cache mounted (expected) -- test studies built on the fly by the "
          "cache-era preprocessing")


def ensure_cache(c):
    """The manifest of the cache `c` resolves to. Missing -> loud failure (traps 6f): every
    recipe since v03 depends on cache-era preprocessing and the decode branch would silently
    train v02 pixels at 5.5x the cost. ALLOW_DECODE_FALLBACK takes it deliberately (c01 only)."""
    if not c.use_cache:
        return None
    cv = cache_version_for(c)
    if cv in cache_manifests:
        return cache_manifests[cv]
    if ALLOW_DECODE_FALLBACK and cache_geom(c)[0] == "c01":
        print(f"  ! use_cache=True but cache {cv} is not mounted -- falling back to per-epoch "
              f"DICOM decode (ALLOW_DECODE_FALLBACK=True)")
        c.use_cache = False
        return None
    raise SystemExit(
        f"use_cache=True but cache {cv} is not mounted (mounted: {sorted(cache_manifests) or 'none'}). "
        f"Attach the matching cache kernels as kernel_sources (c01: rsna-knee-cache-a/-b; "
        f"c02: rsna-knee-cache2-a/-b/-c/-d), or set ALLOW_DECODE_FALLBACK=True to train on the "
        f"v02 decode path deliberately.")


def training_manifest(cache_manifest):
    """Train manifest for one cache (slots, side, mask straight from its manifest; a header scan
    only on the legacy decode path), plus placeholder target rows for imaged studies that are
    not in targets (the local sample). Mutates the module-level `targets`."""
    global targets
    if cache_manifest is not None:
        manifest = cache_manifest[["StudyInstanceUID", *SLOTS, "n_slots", "side", "mask"]].copy()
        print(f"  manifest from cache: {len(manifest)} studies; mean slots "
              f"{manifest.n_slots.mean():.2f}; side resolved {(manifest.side.fillna('') != '').mean():.1%}")
    else:
        train_series_csv = os.path.join(COMP, "train_series.csv")
        series_df = scan_series(train_series_csv, TRAIN_IMG,
                                os.path.join(WORK, "series_scan_train.csv"),
                                max_studies=cfg.smoke_max_studies if cfg.smoke else 0)
        if len(series_df) == 0:
            # Local sample: train_series.csv describes studies we do not have. Fall back to
            # scanning test_series.csv so the smoke test has something to chew on.
            series_df = scan_series(os.path.join(COMP, "test_series.csv"), TRAIN_IMG,
                                    os.path.join(WORK, "series_scan_fallback.csv"))
        manifest = build_manifest(series_df, os.path.join(WORK, "manifest_train.csv"))
    missing = set(manifest.StudyInstanceUID) - set(targets.StudyInstanceUID)
    if missing:
        print(f"  {len(missing)} imaged studies not in targets; adding placeholder "
              f"targets (smoke only)")
        add = pd.DataFrame({"StudyInstanceUID": sorted(missing)})
        add["is_gold"] = 0
        add["report_group"] = "local"
        add["fold"] = 0
        for l in LABELS:
            add[l] = 0.5
        for l in LABELS:
            add[f"w__{l}"] = cfg.weak_weight_floor
        targets = pd.concat([targets, add], ignore_index=True)
    return manifest


def _self_source():
    """The text of this pipeline for the P-31 children: the nbgen-embedded payload inside a notebook, the
    file itself when run as a script (locally / RunPod)."""
    import base64
    import zlib
    if SELF_SOURCE_B64:
        raw = zlib.decompress(base64.b64decode(SELF_SOURCE_B64)).decode("utf-8")
        if hashlib.sha256(raw.encode("utf-8")).hexdigest() != SELF_SOURCE_SHA256:
            raise SystemExit("SELF_SOURCE_B64 sha256 mismatch -- the embedded pipeline payload is corrupt")
        return raw
    path = globals().get("__file__")          # undefined inside a notebook
    if path and os.path.isfile(path):
        with open(path, encoding="utf-8") as f:
            return f.read()
    raise SystemExit("PARALLEL_ARMS needs the pipeline source: build the notebook with src/nbgen.py "
                     "(SELF_SOURCE_B64 is filled when PARALLEL_ARMS is set) or run the .py directly")


def _killpg(proc):
    import signal
    for sig, wait in ((signal.SIGTERM, 30), (signal.SIGKILL, 10)):
        try:
            os.killpg(proc.pid, sig)
            proc.wait(timeout=wait)
            return
        except Exception:
            pass


def _shell(cmd):
    import subprocess
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=20).stdout.strip()
    except Exception as e:
        return f"({type(e).__name__})"


def run_parallel_arms(arms, results):
    """P-31: one child process per arm, one GPU each, this file as the child's script (RSNA_CHILD=1,
    RSNA_ARM=<arm>, CUDA_VISIBLE_DEVICES=<i>, RSNA_TRAIN_ONLY=1). Each child's stdout+stderr goes to
    WORK/<arm>.log -- ipykernel captures Python-level stdout only, so an inherited fd would never reach
    the Kaggle log -- and the parent prints a heartbeat with each log's tail, GPU memory / utilisation
    and host RAM, kills the process groups at the session deadline, and judges each child by its
    ARTEFACTS (`{arm}_fold0_best.pt`), not its exit code (traps 14). Returns True when the children ran
    (the parent then trains and infers nothing), False to fall through to the sequential loop."""
    import subprocess
    import sys
    n_gpu = torch.cuda.device_count()           # NVML-backed: creates no CUDA context in this process
    if not ON_KAGGLE or n_gpu < 2:
        print(f"PARALLEL_ARMS {list(arms)}: {n_gpu} GPU(s) visible, ON_KAGGLE={ON_KAGGLE} -> sequential arm loop")
        return False
    if len(arms) > n_gpu:
        raise SystemExit(f"PARALLEL_ARMS has {len(arms)} arms for {n_gpu} GPUs (two arms on one T4 would OOM)")
    src = _self_source()
    child_py = os.path.join(WORK, "_child.py")
    compile(src, child_py, "exec")
    with open(child_py, "w", encoding="utf-8") as f:
        f.write(src)
    # The children's own runtime guard counts from THEIR start; hand them the remaining budget minus ten
    # minutes for this process to collect and report, and keep a hard deadline of our own behind theirs.
    budget_h = max(0.1, cfg.runtime_limit_hours - elapsed_h() - 0.17)
    deadline = T_START + (cfg.runtime_limit_hours + 0.35) * 3600
    procs = {}
    for i, arm in enumerate(arms):
        env = dict(os.environ)
        env.update(RSNA_CHILD="1", RSNA_ARM=arm, CUDA_VISIBLE_DEVICES=str(i),
                   RSNA_WORKERS=str(max(1, int(cfg.num_workers))), RSNA_TRAIN_ONLY="1",
                   RSNA_RUNTIME_H=f"{budget_h:.2f}", PYTHONUNBUFFERED="1", PYTHONUTF8="1")
        if cfg.smoke:
            # a smoke of the parallel path must exercise the real batch_studies x train_windows memory
            # (P-32) on its handful of studies -- the one thing a 4-window smoke could never reveal
            env["RSNA_SMOKE_FULL_WINDOWS"] = "1"
        log = open(os.path.join(WORK, f"{arm}.log"), "w", encoding="utf-8")
        p = subprocess.Popen([sys.executable, child_py], cwd=WORK, env=env, stdout=log,
                             stderr=subprocess.STDOUT, start_new_session=True)
        procs[arm] = (p, log)
        print(f"  [{arm}] pid {p.pid} on cuda:{i} -> {arm}.log  (child RSNA_RUNTIME_H {budget_h:.2f} h, "
              f"workers {env['RSNA_WORKERS']})", flush=True)

    def tail(arm, n=3):
        try:
            with open(os.path.join(WORK, f"{arm}.log"), encoding="utf-8", errors="replace") as f:
                return f.read().splitlines()[-n:]
        except OSError:
            return []

    t_beat = 0.0
    while any(p.poll() is None for p, _ in procs.values()):
        if time.time() > deadline:
            print(f"  !! parent deadline ({(deadline - T_START) / 3600:.2f} h) -- killing the children; their "
                  f"_last.pt checkpoints survive for a sibling-slug resume (traps 31)", flush=True)
            for p, _ in procs.values():
                if p.poll() is None:
                    _killpg(p)
            break
        if time.time() - t_beat >= 180:
            t_beat = time.time()
            for arm in procs:
                for ln in tail(arm):
                    print(f"  [{arm}] {ln[:220]}")
            gpu = _shell("nvidia-smi --query-gpu=index,memory.used,utilization.gpu --format=csv,noheader")
            mem = _shell("free -g | awk '/Mem/{print $3\"/\"$2\" GB\"}'")
            print(f"  -- heartbeat {elapsed_h():.2f} h | GPU {gpu.replace(chr(10), ' ; ')} | host RAM used/total "
                  f"{mem}", flush=True)
        time.sleep(15)

    import re as _re
    for arm, (p, log) in procs.items():
        log.close()
        rc = p.poll()
        best = os.path.exists(os.path.join(WORK, f"{arm}_fold0_best.pt"))
        last = os.path.exists(os.path.join(WORK, f"{arm}_fold0_last.pt"))
        ep_lines = [ln for ln in tail(arm, 400)
                    if _re.search(r"epoch \d+ EMA score|stopping: runtime guard|FAILED|Error|SWA of last", ln)]
        results[f"{arm}/0"] = {"best": float("nan"), "completed": bool(best and rc == 0)}
        tag = "ok  " if (rc == 0 and best) else "!!  "
        print(f"  {tag}arm {arm}: rc={rc}, _best.pt {'written' if best else 'MISSING'}, _last.pt "
              f"{'present' if last else 'missing'}; last lines: {[ln.strip()[:120] for ln in ep_lines[-2:]]}")
        if not best:
            print(f"      -> {arm} did not finish: resume it in the sibling slug with this output in kernel_sources "
                  f"(traps 31); {arm}.log has the cause")
    print("PARALLEL_ARMS done:", json.dumps(results, indent=1), flush=True)
    return True


results = {}
_parallel_done = False
if mode == "train" and PARALLEL_ARMS and not os.environ.get("RSNA_CHILD"):
    _parallel_done = run_parallel_arms(PARALLEL_ARMS, results)   # P-31: the children train; this process reports
if mode == "train" and _parallel_done:
    ckpt_members = []                 # nothing to infer here: each child stops before Section 9 (RSNA_TRAIN_ONLY)
elif mode == "train":
    # Kaggle only: this script has no `if __name__ == "__main__"` guard, and Windows spawns
    # workers (re-importing __main__) instead of forking. The bug it tests is fork-specific.
    if ON_KAGGLE:
        check_worker_rng()
    base_cfg = replace(cfg)
    for arm_version, overrides in (ARMS or [(cfg.version, {})]):
        # Rebind the module-level `cfg`: out_of_time(), the dataset and the loaders all
        # read the global, so a local copy would silently leave them on the previous arm.
        # Merge, do not double-unpack: an override that sets `folds` (a 5-fold arm) would
        # otherwise be a duplicate keyword argument and raise TypeError. Overrides win.
        _ov = {**({"folds": ARM_FOLDS} if ARMS else {}), **overrides}
        cfg = replace(base_cfg, version=arm_version, **_ov)
        cfg.backbone_dir = resolve_backbone_dir(cfg.backbone)   # an arm may switch family (P-10)
        globals()["cfg"] = cfg
        # Resume is PER ARM (traps 31): copy this arm's mounted `_last.pt` / `_best.pt` into WORK
        # so train_fold continues at epoch+1. Shallow glob, seconds. Smoke never resumes (traps 19).
        if not cfg.smoke:
            for fold in cfg.folds:
                for kind in ("last", "best"):
                    src = find_mounted_checkpoints(cfg.version, kind).get(fold)
                    dst = os.path.join(WORK, f"{cfg.version}_fold{fold}_{kind}.pt")
                    if src and not os.path.exists(dst):
                        shutil.copy(src, dst)
                        print(f"  resume: copied {os.path.basename(src)} into WORK")
        # The cache and the manifest are per ARM: an arm may read a different cache scheme
        # than the default config (c02 arms next to c01 ones), so this cannot happen once
        # before the loop -- that would silently index the default config's cache for every arm.
        manifest = training_manifest(ensure_cache(cfg))
        if ARMS:
            print(f"\n########## arm {arm_version}: {overrides or 'baseline'} "
                  f"| folds {cfg.folds} epochs {cfg.epochs} seed {cfg.seed} ##########")
            print(f"  cache {cache_version_for(cfg)} | window_mode {cfg.window_mode}"
                  + (f" (train {cfg.train_windows}, eval {cfg.eval_windows or 'all'})"
                     if cfg.window_mode == "random" else f" (K {cfg.slices_per_slot})")
                  + f" | head {cfg.head_type} | backbone {cfg.backbone} | img {cfg.img_size}"
                  + f" | batch {cfg.batch_studies} x accum {cfg.grad_accum} | aug {cfg.aug}"
                  + (f" | train_all, swa_last {cfg.swa_last}" if cfg.train_all else ""))
            if cfg.lat_undo:
                n_r = int((manifest["side"].astype(str) == "R").sum())                     if "side" in manifest.columns else 0
                print(f"  lat_undo: {n_r} of {len(manifest)} studies "
                      f"({n_r/max(len(manifest),1):.1%}) de-canonicalised at load time")
        try:
            for fold in cfg.folds:
                if out_of_time():
                    print(f"skipping fold {fold}: out of time")
                    continue
                print(f"\n=== {cfg.version} fold {fold} ===")
                _, best, done = train_fold(fold, manifest, targets, TRAIN_IMG, cfg, device)
                results[f"{cfg.version}/{fold}"] = {"best": best, "completed": done}
                gc.collect()
                if device.type == "cuda":
                    torch.cuda.empty_cache()
        except Exception:
            # One arm failing must not cost the other three -- the Kaggle session is the
            # scarce resource here, not the code. Loud, logged, and on to the next arm.
            print(f"  !! arm {arm_version} FAILED -- continuing with the next arm")
            traceback.print_exc()
            results[f"{arm_version}/failed"] = {"best": float("nan"), "completed": False}
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()

    # The inference below runs for ONE arm. It is a free smoke of the infer path, not a
    # submission -- what gets submitted is kaggle/rsna-knee-infer (traps.md 12c).
    if ARMS:
        cfg = replace(base_cfg, version=PRIMARY_ARM,
                      **{"folds": ARM_FOLDS, **dict(ARMS)[PRIMARY_ARM]})
        cfg.backbone_dir = resolve_backbone_dir(cfg.backbone)
        globals()["cfg"] = cfg
        print(f"\ninference uses PRIMARY_ARM={PRIMARY_ARM}")
    # members are (version, fold, path), the same shape the infer branch builds
    ckpt_members = [(cfg.version, f, os.path.join(WORK, f"{cfg.version}_fold{f}_best.pt"))
                    for f in cfg.folds]
elif mode == "oof_eval":
    # P-12 / P-25 measurement mode: score each member's fold-0 checkpoint on its own held-out
    # studies from the cache with the TTA / eval_windows it would use at inference, so the
    # `_tta_oof.csv` it writes is read by src/blend_check.py exactly like a training OOF file.
    base_cfg = replace(cfg)
    for v, f, p in infer_members:
        s = infer_settings[(v, f)]
        mcfg = replace(base_cfg, version=v)
        apply_settings(mcfg, s, INFER_CACHE_KEYS + INFER_MEMBER_KEYS)   # exact member settings, no smoke clamps
        mcfg.backbone_dir = resolve_backbone_dir(mcfg.backbone)
        # traps 32: a train_all member (P-28) trained on 871 of fold 0's 882 studies -- scoring them
        # would print a flattering "OOF". Such a member is scored on the 58 gold rows only.
        mcfg.train_all = bool(infer_saved_cfg.get((v, f), {}).get("train_all", False))
        if mcfg.train_all:
            print(f"  {v}: trained on every report-labelled study -> scoring the 58 gold rows only")
        globals()["cfg"] = mcfg
        cfg = mcfg
        print(f"\n=== oof_eval {v}/fold{f}: cache {cache_version_for(cfg)}, {s['window_mode']}, "
              f"eval_windows {s['eval_windows'] or 'all'}, tta {s['tta_offsets']}/{s['tta_pool']} ===")
        manifest = training_manifest(ensure_cache(cfg))
        _, va_loader = make_loaders(manifest, targets, TRAIN_IMG, cfg, f)
        model = build_model(cfg, device)
        st = torch.load(p, map_location=device, weights_only=False)
        model.load_state_dict(st["model"])
        t_eval = time.time()
        metrics, table = evaluate(model, va_loader, device, cfg)
        per_label = metrics.pop("per_label", {})
        print(f"  {v}/fold{f}: {metrics}  ({(time.time()-t_eval)/60:.1f} min)")
        if per_label:
            print_per_label(per_label)
        if table is not None:
            out_csv = os.path.join(WORK, f"{v}_fold{f}_tta_oof.csv")
            table.to_csv(out_csv, index=False)
            print(f"  -> {out_csv} ({len(table)} studies)")
        results[f"{v}/{f}"] = {"best": metrics.get("auc_soft", float("nan")), "completed": True}
        del model, st
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
    ckpt_members = []
else:
    results = {f"{v}/{f}": {"best": float("nan"), "completed": True} for v, f, _ in infer_members}
    ckpt_members = list(infer_members)

print("\nfold results:", json.dumps(results, indent=1))
if os.environ.get("RSNA_TRAIN_ONLY") and mode == "train":
    # Off-Kaggle (RunPod) training box: there is no test tree, so stop cleanly here instead of
    # dying at the coverage gate below. The checkpoints in WORK are the deliverable.
    print("RSNA_TRAIN_ONLY is set -- stopping before inference (train-only box)")
    raise SystemExit(0)
if mode == "infer":
    all_done = True                       # every member was verified mounted above
elif mode == "oof_eval":
    all_done = False                      # measurement only; nothing to submit
    print("oof_eval done -- no test prediction in this mode")
else:
    # With ARMS, `results` is keyed "<arm>/<fold>" across every arm, so completion has to be
    # judged on the arm inference will actually use -- otherwise the count never matches
    # len(cfg.folds) and the infer path is silently skipped.
    done_keys = ([k for k in results if str(k).startswith(f"{PRIMARY_ARM}/")]
                 if ARMS else list(results))
    all_done = len(done_keys) == len(cfg.folds) and all(results[k]["completed"] for k in done_keys)
    if _parallel_done:
        all_done = False              # P-31 parent: the children hold the checkpoints; no inference here
print(f"all folds complete: {all_done}  elapsed {elapsed_h():.2f} h")

## Section 9: inference and submission

Ensembling is a **rank mean**, not a probability mean. AUC reads only order, so
averaging probabilities lets whichever fold is most confident dominate, while
averaging ranks combines exactly the information the metric uses.

Inference only runs once every fold has finished. If the runtime guard fired,
the notebook stops here — attach this output as input to a fresh run and it
resumes rather than submitting a half-trained ensemble.

In [ ]:
# ── Section 9: inference ──────────────────────────────────────────────────────
def predict(model, manifest, image_root, cfg, studies, device):
    ds = KneeStudyDataset(manifest, None, image_root, cfg, False, studies)
    # one study per batch always (a training arm's batch_studies must not leak into inference);
    # window-mode items need the collate even at batch 1 (forward_batch's contract)
    dl = DataLoader(ds, batch_size=1, shuffle=False,
                    num_workers=0 if cfg.smoke else cfg.num_workers,
                    collate_fn=collate_windows if getattr(cfg, "window_mode", "fixed") == "random" else None)
    ids, preds = [], []
    model.eval()
    with torch.no_grad():
        for b in dl:
            preds.append(predict_probs(model, b, device, cfg).cpu().numpy())
            ids.extend(b["study"])
    if not preds:
        return pd.DataFrame(columns=["StudyInstanceUID"] + LABELS)
    P = np.concatenate(preds)
    return pd.DataFrame({"StudyInstanceUID": ids,
                         **{l: P[:, i] for i, l in enumerate(LABELS)}})


def rank_mean(frames):
    """Average percentile ranks across folds -- the operation macro-AUC actually reads."""
    base = frames[0][["StudyInstanceUID"]].copy()
    for lab in LABELS:
        acc = np.zeros(len(base))
        for f in frames:
            acc += f[lab].rank(pct=True).to_numpy()
        base[lab] = acc / len(frames)
    return base


sub_path = os.path.join(WORK, "submission.csv")
sample_path = os.path.join(COMP, "sample_submission.csv")
ref = pd.read_csv(sample_path)

if not all_done:
    print("training incomplete -- skipping inference.")
    print("Attach this notebook's output as input to a new run to resume.")
else:
    # Deliberately NO placeholder file: if anything below raises, Kaggle reports a
    # missing submission (visible), instead of scoring a silent 0.500 (invisible).
    for stale in (sub_path, "/kaggle/working/submission.csv" if ON_KAGGLE else None):
        if stale and os.path.exists(stale):
            os.remove(stale)

    t_inf = time.time()
    test_series_df = scan_series(os.path.join(COMP, "test_series.csv"), TEST_IMG,
                                 os.path.join(WORK, "series_scan_test.csv"))
    test_manifest = build_manifest(test_series_df,
                                   os.path.join(WORK, "manifest_test.csv"))
    all_test = pd.read_csv(os.path.join(COMP, "test.csv")).StudyInstanceUID.tolist()
    with_slots = set(test_manifest.loc[test_manifest.n_slots > 0, "StudyInstanceUID"])
    test_studies = [s for s in all_test if s in with_slots]    # imaged AND has a slot
    coverage = len(test_studies) / max(len(all_test), 1)
    print(f"  test studies: {len(all_test)} listed, {len(test_studies)} imaged "
          f"({coverage:.1%}); scan+manifest {time.time()-t_inf:.0f}s")
    print("  slot fill on test:",
          {s: round(float((test_manifest[s] != '').mean()), 3) for s in SLOTS})
    # Loud failure beats a silent constant submission: a scoring error is visible on
    # the submissions page, a 0.500 looks like a bad model.
    if coverage < 0.9:
        raise SystemExit(f"only {coverage:.1%} of test studies have images under "
                         f"{TEST_IMG} -- refusing to submit constants")

    # ---- decode once PER GEOMETRY GROUP, predict with every member (P-18 / P-21 / P-25) ------
    # A test study is never in the mounted cache, so each member used to re-decode the whole
    # test set (~1.5-2 s/study). Members that share every CACHE key form a group; each group's
    # test arrays are built ONCE with build_study_array -- the cache builder's own function, so
    # a test study is preprocessed exactly like a cached training study -- stored under the
    # system temp dir (NOT WORK: 5-8 MB/study must not become kernel output), registered in
    # CACHE_INDEX[version] so KneeStudyDataset takes the same read branch it takes in training,
    # and deleted once the group's members have predicted (two schemes = two footprints).
    import shutil

    def decode_once(group_cfg, studies, manifest_df):
        version = cache_version_for(group_cfg)
        test_cache_dir = os.path.join(tempfile.gettempdir(), "rsna_test_cache", version)
        os.makedirs(test_cache_dir, exist_ok=True)

        class _BuildOnce(Dataset):
            def __init__(self, manifest, studies):
                self.m = manifest.set_index("StudyInstanceUID")
                self.s = list(studies)

            def __len__(self):
                return len(self.s)

            def __getitem__(self, i):
                study = self.s[i]
                arr, mask = build_study_array(study, self.m.loc[study], TEST_IMG, group_cfg)
                path = os.path.join(test_cache_dir, f"{study}.npy")
                np.save(path, arr)
                return study, path, "".join("1" if v > 0 else "0" for v in mask)

        t_dec = time.time()
        masks, index = {}, {}
        dec_loader = DataLoader(_BuildOnce(manifest_df, studies), batch_size=1, shuffle=False,
                                num_workers=0 if group_cfg.smoke else group_cfg.num_workers,
                                collate_fn=lambda b: b[0])
        for k, (study, path, mk) in enumerate(dec_loader):
            index[study] = path
            masks[study] = mk
            if (k + 1) in (10, 100) or (k + 1) % 500 == 0:
                dt = time.time() - t_dec
                print(f"    decoded {k+1}/{len(studies)} test studies in {dt:.0f}s "
                      f"({dt/(k+1):.2f} s/study) -> ETA {dt/(k+1)*len(studies)/60:.0f} min")
        CACHE_INDEX[version] = index
        n_bytes = sum(os.path.getsize(index[s]) for s in studies[:50]) * len(studies) / max(min(50, len(studies)), 1)
        print(f"  decode-once [{version}]: {len(masks)} test studies -> {test_cache_dir} in "
              f"{(time.time()-t_dec)/60:.1f} min (~{n_bytes/1e9:.1f} GB)")
        # Verify by equality, not by absence of errors (traps 6d/6e): rebuild a few studies on
        # the fly and compare with what every member of the group is about to read.
        _chk = manifest_df.set_index("StudyInstanceUID")
        for study in studies[:3]:
            arr, mask = build_study_array(study, _chk.loc[study], TEST_IMG, group_cfg)
            mk = "".join("1" if v > 0 else "0" for v in mask)
            if not (np.array_equal(arr, np.load(index[study])) and mk == masks[study]):
                raise SystemExit(f"decode-once mismatch on {study}: the stored array or mask "
                                 f"differs from a fresh build -- refusing to predict")
        print(f"  decode-once verified [{version}]: {min(3, len(studies))} studies rebuilt, identical")
        return version, masks, test_cache_dir

    member_list = []                      # (version, fold, path, settings)
    for v, fold, ck in ckpt_members:
        if not ck or not os.path.exists(ck):
            print(f"  {v}/fold{fold}: no checkpoint, skipped")
            continue
        s = infer_settings.get((v, fold))
        if s is None:                     # train mode: this run's own checkpoints
            st0 = torch.load(ck, map_location="cpu", weights_only=False)
            s = member_settings(st0.get("config", {}), v)
            del st0
        member_list.append((v, fold, ck, s))
    geometry_groups = {}
    for item in member_list:
        geometry_groups.setdefault(cache_signature(item[3]), []).append(item)
    print(f"  {len(member_list)} members in {len(geometry_groups)} geometry group(s)")

    frames, member_tags = [], []
    cfg_snapshot = replace(cfg)
    for sig, members in geometry_groups.items():
        apply_settings(cfg, members[0][3], INFER_CACHE_KEYS)
        group_version, tmp_dir = cache_version_for(cfg), None
        if cfg.use_cache and test_studies:
            group_version, masks, tmp_dir = decode_once(cfg, test_studies, test_manifest)
            test_manifest["mask"] = test_manifest.StudyInstanceUID.map(masks).fillna("")
        for v, fold, ck, s in members:
            prev = apply_settings(cfg, s, INFER_MEMBER_KEYS)
            st = torch.load(ck, map_location=device, weights_only=False)
            m = build_model(s, device)
            m.load_state_dict(st["model"])
            t_f = time.time()
            frames.append(predict(m, test_manifest, TEST_IMG, cfg, test_studies, device))
            member_tags.append(f"{v}/fold{fold}")
            dt = time.time() - t_f
            how = (f"windows eval {s['eval_windows'] or 'all'}" if s["window_mode"] == "random"
                   else f"K {s['slices_per_slot']}, tta {s['tta_offsets']}/{s['tta_pool']}, {s['stack_mode']}")
            print(f"  {v}/fold{fold} ({s['backbone']}, {s['head_type']}, {how}, {group_version}): "
                  f"predicted {len(frames[-1])} studies in {dt:.0f}s "
                  f"({dt/max(len(frames[-1]),1)*100:.0f} s per 100 studies) "
                  f"[epoch {st.get('epoch')}, score {st.get('score')}, ema {st.get('ema')}]")
            apply_settings(cfg, prev, INFER_MEMBER_KEYS)
            del m, st
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()
        if tmp_dir:
            shutil.rmtree(tmp_dir, ignore_errors=True)
            CACHE_INDEX.pop(group_version, None)
    apply_settings(cfg, {k: getattr(cfg_snapshot, k) for k in INFER_CACHE_KEYS}, INFER_CACHE_KEYS)

    if not frames:
        raise SystemExit("no checkpoints produced predictions -- refusing to submit "
                         "constants")
    if len(frames) > 1 and len(frames[0]) > 3:
        # Two members that agree perfectly are one model counted twice; print the rank
        # correlation so the blend's diversity is on the record (P-21 measured 0.773 on OOF).
        for i in range(len(frames)):
            for j in range(i + 1, len(frames)):
                rho = float(np.mean([frames[i][l].corr(frames[j][l], method="spearman")
                                     for l in LABELS]))
                print(f"  rank correlation {member_tags[i]} vs {member_tags[j]}: {rho:.3f}")
    if INFER_BLEND == "by_version":
        by_version = {}
        for tag, f in zip(member_tags, frames):
            by_version.setdefault(tag.split("/")[0], []).append(f)
        sub = rank_mean([rank_mean(fs) for fs in by_version.values()])
        print("  blend: by_version -> " + ", ".join(f"{v} ({len(fs)} fold{'s' if len(fs) != 1 else ''})"
                                                  for v, fs in by_version.items()))
    else:
        sub = rank_mean(frames)
        print(f"  blend: flat over {len(frames)} members")

    # Any study we could not image must still appear, or the submission is rejected.
    sub = ref[["StudyInstanceUID"]].merge(sub, on="StudyInstanceUID", how="left")
    n_filled = int(sub[LABELS[0]].isna().sum())
    for l in LABELS:
        sub[l] = sub[l].fillna(0.5)
    sub = sub[["StudyInstanceUID"] + LABELS]

    assert list(sub.columns) == list(ref.columns), "column mismatch vs sample_submission"
    assert len(sub) == len(ref), f"row count {len(sub)} != {len(ref)}"
    assert (sub.StudyInstanceUID.to_numpy() == ref.StudyInstanceUID.to_numpy()).all(), \
        "row order differs from sample_submission"
    assert np.isfinite(sub[LABELS].to_numpy()).all(), "non-finite predictions"
    n_const = int((sub[LABELS].std(axis=0) < 1e-9).sum())
    if n_const > len(LABELS) // 2 and len(sub) > 3:
        raise SystemExit(f"{n_const}/12 labels are constant across {len(sub)} studies "
                         f"-- model or inputs are broken, refusing to submit")

    sub.to_csv(sub_path, index=False)
    if ON_KAGGLE:
        sub.to_csv("/kaggle/working/submission.csv", index=False)
    print(f"\nwrote {sub_path}  rows={len(sub)}  filled 0.5 for {n_filled}  "
          f"range=[{sub[LABELS].to_numpy().min():.3f}, "
          f"{sub[LABELS].to_numpy().max():.3f}]  constant labels {n_const}  "
          f"inference total {(time.time()-t_inf)/60:.1f} min")
    print(sub.head(3).to_string(index=False))

print(f"\ntotal elapsed {elapsed_h():.2f} h")